In [ ]:
#| default_exp model_selection

## Hyperparameters tuning methods for machine learning models

In [ ]:
#| export

import numpy as np
from typing import List, Dict, Optional, Callable, Tuple, Any, Union

#------------------------------------------------------------------------------
# Parametric Time Series Split
#------------------------------------------------------------------------------
class SplitTimeSeries:

    def __init__(self,
                 n_splits: int,
                 test_size: int,
                 step_size: Optional[int] = None
                 ) -> None:

        """
        A time series cross-validator that generates train/test splits with a fixed test size and a configurable step size.

        Parameters
        ----------
        n_splits : int
            Number of splits to generate.
        test_size : int
            The number of samples in each test set.
        step_size : int, optional
            The number of samples to move the test set forward for each split. If None, it defaults to the test_size, meaning non-overlapping test sets.
        """
        self.test_size = test_size
        self.step_size = test_size if step_size is None else step_size
        self.n_splits = n_splits

    def split(self, X):
        n_samples = len(X)
        split_starts = []
        # Start the last test set at the last possible position
        last_test_start = n_samples - self.test_size
        # Build test starts, moving backward with fixed step_size
        current = last_test_start
        while current >= 0:
            split_starts.append(current)
            current -= self.step_size
        split_starts = split_starts[::-1]  # Reverse to start from earliest

        # Use only the last n_splits
        split_starts = split_starts[-self.n_splits:]

        for test_start in split_starts:
            test_end = test_start + self.test_size
            train_index = np.arange(0, test_start)
            test_index = np.arange(test_start, test_end)
            yield train_index, test_index


In [ ]:
#| hide
from fastcore.docments import docments, DocmentTbl
from nbdev.showdoc import *

In [ ]:
#| echo: false
show_doc(SplitTimeSeries)

---

[source](https://github.com/mustafaslanCoto/peshbeen/blob/main/peshbeen/model_selection.py#L15){target="_blank" style="float:right; font-size:smaller"}

### SplitTimeSeries

```python

def SplitTimeSeries(
    n_splits:int, # Number of splits to generate.
    test_size:int, # The number of samples in each test set.
    step_size:Optional=None, # The number of samples to move the test set forward for each split. If None, it defaults to the test_size, meaning non-overlapping test sets.
)->None:


```

*A time series cross-validator that generates train/test splits with a fixed test size and a configurable step size.*

In [ ]:
#| echo: false
DocmentTbl(SplitTimeSeries)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| n_splits | int |  | Number of splits to generate. |
| test_size | int |  | The number of samples in each test set. |
| step_size | Optional | None | The number of samples to move the test set forward for each split. If None, it defaults to the test_size, meaning non-overlapping test sets. |
| **Returns** | **None** |  |  |

In [ ]:
#| export


from pyexpat import model
import pandas as pd
import numpy as np
from scipy import stats
from numba import jit
from scipy.stats import boxcox
from scipy.special import inv_boxcox
from sklearn.linear_model import LinearRegression
from numba import jit
##Stationarity Check
from statsmodels.tsa.stattools import adfuller, kpss
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statistics import NormalDist
import warnings
warnings.filterwarnings("ignore")
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tqdm import tqdm_notebook
from itertools import product
from typing import List, Dict, Optional, Callable, Tuple, Any, Union

def hyperopt_tune(
    model: object,
    df: pd.DataFrame,
    cv_split: int,
    test_size: int,
    eval_metric: Callable,
    param_space: dict,
    step_size: int = None,
    eval_num=100,
    verbose=False
    ) -> Tuple[Dict[str, Any], List[int], List[str]]:

    """
    Tune forecasting model hyperparameters using time series cross-validation and hyperopt.

    Parameters
    ----------
    model : object
        Forecasting model object with .fit and .forecast methods and relevant attributes.
    df : pd.DataFrame
        Time series data with a datetime index and a target column and optionally exogenous features.
    cv_split : int
        Number of cross-validation splits.
    test_size : int
        Number of samples in each test set.
    eval_metric : Callable
        Evaluation metric function.
    step_size : int, optional
        Step size to move the test window forward in each split.
    param_space : dict
        Hyperparameter search space for the forecasting model.
    eval_num : int, optional
        Number of hyperparameter combinations to evaluate. Default is 100.
    verbose : bool, optional
        Whether to print the evaluation metric for each hyperparameter combination. Default is False.
    
    Returns
    -------
    Tuple[Dict[str, Any], List[int], List[str]]
        A tuple containing the best hyperparameters, selected lags, and selected transforms.
    """
    tscv = SplitTimeSeries(n_splits=cv_split, test_size=test_size, step_size=step_size)

    def _set_model_params(params):
        # Handle special model parameters that are not passed to model constructor
        # and must be set directly on the forecasting model object
        if "lags" in params:
            if isinstance(params["lags"], int):
                model.n_lag = list(range(1, params["lags"] + 1))
            elif isinstance(params["lags"], list):
                model.n_lag = params["lags"]
        if "box_cox" in params:
            model.box_cox = params["box_cox"]
        if "box_cox_lmda" in params:
            model.lamda = params["box_cox_lmda"]
        if "box_cox_biasadj" in params:
            model.biasadj = params["box_cox_biasadj"]

    def _get_model_params_for_fit(params):
        # Exclude special parameters that should not be passed to the model constructor
        skip_keys = {"box_cox", "lags", "box_cox_lmda", "box_cox_biasadj"}

        return {k: v for k, v in params.items() if k not in skip_keys}
    

    def objective(params):
        _set_model_params(params)
        if isinstance(model.model, LinearRegression):
            # For LinearRegression, we don't need to set model_params
            model_params = None
        else:
            # For other models, get the parameters to set
            model_params = _get_model_params_for_fit(params)

        metrics = []
        for train_index, test_index in tscv.split(df):
            train, test = df.iloc[train_index], df.iloc[test_index]
            x_test = test.drop(columns=[model.target_col])
            y_test = np.array(test[model.target_col])

            if model_params is not None:
                model.model.set_params(**model_params)
            model.fit(train)
            
            y_pred = model.forecast(len(y_test), x_test)

            #Evaluate using the specified metric
            if eval_metric.__name__ in ["MASE", "SMAE", "SRMSE", "RMSSE"]:
                score = eval_metric(y_test,
                                    y_pred,
                                    train[model.target_col])
            else:
                score = eval_metric(y_test,y_pred)
            metrics.append(score)

        mean_score = np.mean(metrics)
        if verbose:
            print("Score:", mean_score)
        return {"loss": mean_score, "status": STATUS_OK}

    trials = Trials()
    best_hyperparams = fmin(
        fn=objective,
        space=param_space,
        algo=tpe.suggest,
        max_evals=eval_num,
        trials=trials,
    )

    # if lags are in the param space, extract the best lags and extract remain parameters for the model
    if "lags" in param_space:
        best_lags = space_eval(param_space, best_hyperparams)["lags"]
        model_parameters = {k: v for k, v in space_eval(param_space, best_hyperparams).items() if k != "lags"}
    else:
        best_lags = None
        model_parameters = space_eval(param_space, best_hyperparams)

    # model.tuned_params = [
    #     space_eval(param_space, {k: v[0] for k, v in t["misc"]["vals"].items()})
    #     for t in trials.trials]

    return model_parameters, best_lags


/Users/aslanm/Desktop/my_desk/peshbeen/.venv/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
#| echo: false
show_doc(hyperopt_tune)

---

[source](https://github.com/mustafaslanCoto/peshbeen/blob/main/peshbeen/model_selection.py#L85){target="_blank" style="float:right; font-size:smaller"}

### hyperopt_tune

```python

def hyperopt_tune(
    model:object, # Forecasting model object with .fit and .forecast methods and relevant attributes.
    df:DataFrame, # Time series data with a datetime index and a target column and optionally exogenous features.
    cv_split:int, # Number of cross-validation splits.
    test_size:int, # Number of samples in each test set.
    eval_metric:Callable, # Evaluation metric function.
    param_space:dict, # Hyperparameter search space for the forecasting model.
    step_size:int=None, # Step size to move the test window forward in each split.
    eval_num:int=100, # Number of hyperparameter combinations to evaluate. Default is 100.
    verbose:bool=False, # Whether to print the evaluation metric for each hyperparameter combination. Default is False.
)->Tuple: # A tuple containing the best hyperparameters, selected lags, and selected transforms.


```

*Tune forecasting model hyperparameters using time series cross-validation and hyperopt.*

In [ ]:
#| echo: false
DocmentTbl(hyperopt_tune)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | object |  | Forecasting model object with .fit and .forecast methods and relevant attributes. |
| df | DataFrame |  | Time series data with a datetime index and a target column and optionally exogenous features. |
| cv_split | int |  | Number of cross-validation splits. |
| test_size | int |  | Number of samples in each test set. |
| eval_metric | Callable |  | Evaluation metric function. |
| param_space | dict |  | Hyperparameter search space for the forecasting model. |
| step_size | int | None | Step size to move the test window forward in each split. |
| eval_num | int | 100 | Number of hyperparameter combinations to evaluate. Default is 100. |
| verbose | bool | False | Whether to print the evaluation metric for each hyperparameter combination. Default is False. |
| **Returns** | **Tuple** |  | **A tuple containing the best hyperparameters, selected lags, and selected transforms.** |

In [ ]:
#| hide

from peshbeen.datasets import load_wales_admissions
from peshbeen.models import ml_forecaster
from peshbeen.metrics import WMAPE, MAE, RMSE
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
load_wales_admissions["day_of_week"] = load_wales_admissions.index.dayofweek
load_wales_admissions["month"] = load_wales_admissions.index.month
# split the data into train and test sets
train = load_wales_admissions[:-30]
test = load_wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
from sklearn.linear_model import LinearRegression
ml_linear = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables)
ml_linear.fit(train)
# ml_linear.data_prep(train)
forecasts = ml_linear.forecast(H=30, exog=test[cat_variables])

In [ ]:
#| hide
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll import scope
lgb_param_space={'learning_rate': hp.uniform('learning_rate', 0.001, 0.6),
            'num_leaves': scope.int(hp.quniform('num_leaves', 10, 200, 1)),
           'max_depth':scope.int(hp.quniform('max_depth', 2, 18, 1)),
            'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1),
            'feature_fraction': hp.uniform('feature_fraction', 0.5, 1),
           'min_data_in_leaf': scope.int(hp.quniform ('min_data_in_leaf', 5, 100, 1)), 
            'lambda_l2' : hp.uniform('lambda_l2', 0,10),
           'lambda_l1' : hp.uniform('lambda_l1', 0, 10),
            'min_gain_to_split':hp.uniform('min_gain_to_split', 0, 20),
            "max_bin": scope.int(hp.quniform('max_bin', 100, 350, 1)),
           'top_rate' : hp.quniform('top_rate', 0.05, 0.4, 0.0001),
            'other_rate' : hp.quniform('other_rate', 0.05, 0.3, 0.0001),
           'num_iterations': scope.int(hp.quniform("num_iterations", 30, 700, 1)),
           'top_k': scope.int(hp.quniform('top_k', 8, 30, 1)),
           'lags': hp.choice("lags", [[1,2,3,4,5], [1,4,7], [1,2,3,4,5,6,7], [1,2,3,4,5,6,7,14], [1,2,3,4,5,6,7,14,21], [1,2,3]]),
                "seed":0}
best_params= hyperopt_tune(model=ml_linear, df=train, cv_split=5, step_size=10,
                                        test_size=30, eval_metric=RMSE, eval_num=5,
                                        param_space=lgb_param_space)

100%|██████████| 5/5 [00:07<00:00,  1.44s/trial, best loss: 171.8634357225871] 


In [ ]:
#| export

import optuna
def optuna_tune(
    model: object,
    df: pd.DataFrame,
    cv_split: int,
    test_size: int,
    eval_metric: Callable,
    param_space: Dict[str, Any],
    step_size: int = None,
    eval_num: int = 100,
    verbose: bool = False,
) -> Tuple[Dict[str, Any], Any]:
    """
    Tune forecasting model hyperparameters using time series cross-validation and Optuna.
 
    Parameters
    ----------
    model : object
        Forecasting model with .fit and .forecast methods.
    df : pd.DataFrame
        Time series data (datetime index, target column, optional exogenous features).
    cv_split : int
        Number of cross-validation splits.
    test_size : int
        Number of samples in each test fold.
    eval_metric : Callable
        Metric function to minimise.
    param_space : dict
        Each value must be a callable that accepts an Optuna `trial` and returns a value.
    step_size : int, optional
        Step size between CV folds.
    eval_num : int, optional
        Number of Optuna trials. Default 100.
    verbose : bool, optional
        Print score for every trial. Default False.
 
    Returns
    -------
    Tuple[Dict[str, Any], Any]
        Best hyperparameters and best lags (if 'lags' is in param_space).

    """
    tscv = SplitTimeSeries(n_splits=cv_split, test_size=test_size, step_size=step_size)
 
    _skip = {"box_cox", "lags", "box_cox_lmda", "box_cox_biasadj"}
 
    def _set_model_params(params: dict):
        if "lags" in params:
            lags = params["lags"]
            model.n_lag = list(range(1, lags + 1)) if isinstance(lags, int) else list(lags)
        if "box_cox"         in params: model.box_cox = params["box_cox"]
        if "box_cox_lmda"    in params: model.lamda   = params["box_cox_lmda"]
        if "box_cox_biasadj" in params: model.biasadj = params["box_cox_biasadj"]
 
    def _fit_params(params: dict) -> Optional[dict]:
        if isinstance(model.model, LinearRegression):
            return None
        return {k: v for k, v in params.items() if k not in _skip}
 
    def objective(trial: optuna.Trial) -> float:
        params = {name: suggest_fn(trial) for name, suggest_fn in param_space.items()}
 
        _set_model_params(params)
        fit_params = _fit_params(params)
 
        scores = []
        for train_idx, test_idx in tscv.split(df):
            train, test = df.iloc[train_idx], df.iloc[test_idx]
            x_test = test.drop(columns=[model.target_col])
            y_test = np.array(test[model.target_col])
 
            if fit_params is not None:
                model.model.set_params(**fit_params)
            model.fit(train)
            y_pred = model.forecast(len(y_test), x_test)
 
            if eval_metric.__name__ in ("MASE", "SMAE", "SRMSE", "RMSSE"):
                score = eval_metric(y_test, y_pred, train[model.target_col])
            else:
                score = eval_metric(y_test, y_pred)
            scores.append(score)
 
        mean_score = float(np.mean(scores))
        if verbose:
            print(f"Trial {trial.number:>4d} | score={mean_score:.6f} | {params} ")
        return mean_score
 
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=eval_num)
 
    best = study.best_params
 
    if "lags" in param_space:
        best_lags = best.pop("lags")
        if not isinstance(best_lags, list):
            best_lags = list(best_lags)
        model_parameters = best
    else:
        best_lags = None
        model_parameters = best
 
    return model_parameters, best_lags

In [ ]:
#| echo: false
show_doc(optuna_tune)

---

[source](https://github.com/mustafaslanCoto/peshbeen/blob/main/peshbeen/model_selection.py#L211){target="_blank" style="float:right; font-size:smaller"}

### optuna_tune

```python

def optuna_tune(
    model:object, # Forecasting model with .fit and .forecast methods.
    df:DataFrame, # Time series data (datetime index, target column, optional exogenous features).
    cv_split:int, # Number of cross-validation splits.
    test_size:int, # Number of samples in each test fold.
    eval_metric:Callable, # Metric function to minimise.
    param_space:Dict, # Each value must be a callable that accepts an Optuna `trial` and returns a value.
    step_size:int=None, # Step size between CV folds.
    eval_num:int=100, # Number of Optuna trials. Default 100.
    verbose:bool=False, # Print score for every trial. Default False.
)->Tuple: # Best hyperparameters and best lags (if 'lags' is in param_space).


```

*Tune forecasting model hyperparameters using time series cross-validation and Optuna.*

In [ ]:
#| echo: false
DocmentTbl(optuna_tune)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | object |  | Forecasting model with .fit and .forecast methods. |
| df | DataFrame |  | Time series data (datetime index, target column, optional exogenous features). |
| cv_split | int |  | Number of cross-validation splits. |
| test_size | int |  | Number of samples in each test fold. |
| eval_metric | Callable |  | Metric function to minimise. |
| param_space | Dict |  | Each value must be a callable that accepts an Optuna `trial` and returns a value. |
| step_size | int | None | Step size between CV folds. |
| eval_num | int | 100 | Number of Optuna trials. Default 100. |
| verbose | bool | False | Print score for every trial. Default False. |
| **Returns** | **Tuple** |  | **Best hyperparameters and best lags (if 'lags' is in param_space).** |

In [ ]:
#| hide
lgb_param_space = {
    "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
    "num_leaves":        lambda t: t.suggest_int("num_leaves", 10, 200),
    "max_depth":         lambda t: t.suggest_int("max_depth", 2, 18),
    "bagging_fraction":  lambda t: t.suggest_float("bagging_fraction", 0.5, 1.0),
    "feature_fraction":  lambda t: t.suggest_float("feature_fraction", 0.5, 1.0),
    "min_data_in_leaf":  lambda t: t.suggest_int("min_data_in_leaf", 5, 100),
    "lambda_l2":         lambda t: t.suggest_float("lambda_l2", 0.0, 10.0),
    "lambda_l1":         lambda t: t.suggest_float("lambda_l1", 0.0, 10.0),
    "min_gain_to_split": lambda t: t.suggest_float("min_gain_to_split", 0.0, 20.0),
    "max_bin":           lambda t: t.suggest_int("max_bin", 100, 350),
    "top_rate":          lambda t: t.suggest_float("top_rate", 0.05, 0.4),
    "other_rate":        lambda t: t.suggest_float("other_rate", 0.05, 0.3),
    "num_iterations":    lambda t: t.suggest_int("num_iterations", 30, 700),
    "top_k":             lambda t: t.suggest_int("top_k", 8, 30),
    "lags":              lambda t: t.suggest_categorical(
                             "lags", [
                                 (1,2,3,4,5),
                                 (1,4,7),
                                 (1,2,3,4,5,6,7),
                                 (1,2,3,4,5,6,7,14),
                                 (1,2,3,4,5,6,7,14,21),
                                 (1,2,3),
                             ]),
                             
    "seed":              lambda t: 0,   # fixed, not sampled
}

best_params, best_lags = optuna_tune(
    model=ml_linear,
    df=train,
    cv_split=5,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=4,
    param_space=lgb_param_space, verbose=True
)

[I 2026-03-15 22:41:14,582] A new study created in memory with name: no-name-810074dc-3f82-4b0a-b5a0-aefeaf9acbb0
[I 2026-03-15 22:41:16,798] Trial 0 finished with value: 180.02644487050617 and parameters: {'learning_rate': 0.1878110291220951, 'num_leaves': 116, 'max_depth': 16, 'bagging_fraction': 0.922982726976757, 'feature_fraction': 0.7573026058730179, 'min_data_in_leaf': 39, 'lambda_l2': 8.118781910395136, 'lambda_l1': 3.901667879201262, 'min_gain_to_split': 6.4229566243228975, 'max_bin': 219, 'top_rate': 0.3157807313782061, 'other_rate': 0.11117154991684515, 'num_iterations': 241, 'top_k': 25, 'lags': (1, 4, 7)}. Best is trial 0 with value: 180.02644487050617.


Trial    0 | score=180.026445 | {'learning_rate': 0.1878110291220951, 'num_leaves': 116, 'max_depth': 16, 'bagging_fraction': 0.922982726976757, 'feature_fraction': 0.7573026058730179, 'min_data_in_leaf': 39, 'lambda_l2': 8.118781910395136, 'lambda_l1': 3.901667879201262, 'min_gain_to_split': 6.4229566243228975, 'max_bin': 219, 'top_rate': 0.3157807313782061, 'other_rate': 0.11117154991684515, 'num_iterations': 241, 'top_k': 25, 'lags': (1, 4, 7), 'seed': 0} 


[I 2026-03-15 22:41:19,783] Trial 1 finished with value: 182.08352343610028 and parameters: {'learning_rate': 0.2093258800382745, 'num_leaves': 98, 'max_depth': 10, 'bagging_fraction': 0.756050773769235, 'feature_fraction': 0.8591033172871152, 'min_data_in_leaf': 12, 'lambda_l2': 9.560783389568112, 'lambda_l1': 8.57179074277485, 'min_gain_to_split': 0.4156215128722929, 'max_bin': 319, 'top_rate': 0.1297792133517305, 'other_rate': 0.24097417108664448, 'num_iterations': 282, 'top_k': 16, 'lags': (1, 2, 3, 4, 5)}. Best is trial 0 with value: 180.02644487050617.


Trial    1 | score=182.083523 | {'learning_rate': 0.2093258800382745, 'num_leaves': 98, 'max_depth': 10, 'bagging_fraction': 0.756050773769235, 'feature_fraction': 0.8591033172871152, 'min_data_in_leaf': 12, 'lambda_l2': 9.560783389568112, 'lambda_l1': 8.57179074277485, 'min_gain_to_split': 0.4156215128722929, 'max_bin': 319, 'top_rate': 0.1297792133517305, 'other_rate': 0.24097417108664448, 'num_iterations': 282, 'top_k': 16, 'lags': (1, 2, 3, 4, 5), 'seed': 0} 


[I 2026-03-15 22:41:21,884] Trial 2 finished with value: 180.99746878814793 and parameters: {'learning_rate': 0.08351041271097141, 'num_leaves': 103, 'max_depth': 5, 'bagging_fraction': 0.9854649689144603, 'feature_fraction': 0.5782078479663444, 'min_data_in_leaf': 26, 'lambda_l2': 5.421807797886782, 'lambda_l1': 4.1306981744267945, 'min_gain_to_split': 8.607543449536152, 'max_bin': 318, 'top_rate': 0.1169402205731565, 'other_rate': 0.07805941281993221, 'num_iterations': 404, 'top_k': 16, 'lags': (1, 2, 3, 4, 5, 6, 7, 14, 21)}. Best is trial 0 with value: 180.02644487050617.


Trial    2 | score=180.997469 | {'learning_rate': 0.08351041271097141, 'num_leaves': 103, 'max_depth': 5, 'bagging_fraction': 0.9854649689144603, 'feature_fraction': 0.5782078479663444, 'min_data_in_leaf': 26, 'lambda_l2': 5.421807797886782, 'lambda_l1': 4.1306981744267945, 'min_gain_to_split': 8.607543449536152, 'max_bin': 318, 'top_rate': 0.1169402205731565, 'other_rate': 0.07805941281993221, 'num_iterations': 404, 'top_k': 16, 'lags': (1, 2, 3, 4, 5, 6, 7, 14, 21), 'seed': 0} 


[I 2026-03-15 22:41:23,535] Trial 3 finished with value: 186.04262108634143 and parameters: {'learning_rate': 0.16353808801188338, 'num_leaves': 100, 'max_depth': 13, 'bagging_fraction': 0.821273527005594, 'feature_fraction': 0.9719409310714264, 'min_data_in_leaf': 88, 'lambda_l2': 7.224245383782634, 'lambda_l1': 2.946251540878121, 'min_gain_to_split': 4.666510242423563, 'max_bin': 288, 'top_rate': 0.30155637907539234, 'other_rate': 0.19668811744492015, 'num_iterations': 366, 'top_k': 8, 'lags': (1, 2, 3, 4, 5, 6, 7, 14)}. Best is trial 0 with value: 180.02644487050617.


Trial    3 | score=186.042621 | {'learning_rate': 0.16353808801188338, 'num_leaves': 100, 'max_depth': 13, 'bagging_fraction': 0.821273527005594, 'feature_fraction': 0.9719409310714264, 'min_data_in_leaf': 88, 'lambda_l2': 7.224245383782634, 'lambda_l1': 2.946251540878121, 'min_gain_to_split': 4.666510242423563, 'max_bin': 288, 'top_rate': 0.30155637907539234, 'other_rate': 0.19668811744492015, 'num_iterations': 366, 'top_k': 8, 'lags': (1, 2, 3, 4, 5, 6, 7, 14), 'seed': 0} 


## Feature selection methods for univariate time series models

In [ ]:
#| export

#------------------------------------------------------------------------------
# Feature Selection Algorithms
# ------------------------------------------------------------------------------
 
def forward_feature_selection(
    model: object,
    df: pd.DataFrame,
    cv_split: int,
    H: int,
    step_size: Optional[int] = None,
    metrics: Union[Callable, List[Callable]] = None,
    lags_to_consider: Optional[int] = None,
    candidate_features: Optional[List[str]] = None,
    transformations: Optional[List] = None,
    starting_lags: Optional[List[int]] = None,
    starting_transforms: Optional[List] = None,
    best_start_score: Optional[List[float]] = None,
    verbose=False,
):
    """
    Forward stepwise feature selection for `ml_forecaster` models.
 
    At each iteration every remaining candidate (lag, exogenous column, or
    lag-transform) is tested individually by adding it to the current best
    feature set.  The candidate that produces the largest cross-validation
    improvement is permanently added.  The loop continues until no remaining
    candidate improves any of the evaluation metrics.
 
    Parameters
    ----------
    model : object
        A *configured but unfitted* `ml_forecaster` instance.  The function works exclusively on deep copies and never mutates the object passed in.
    df : pd.DataFrame
        Full training DataFrame. Must contain the target column and any candidate exogenous columns.
    cv_split : int
        Number of time-series cross-validation folds.
    H : int
        Forecast horizon (test window size for each fold).
    step_size : int, optional
        Step size between consecutive CV folds.  If `None` (default) the step equals `H`, producing non-overlapping folds — consistent with the default behaviour of `ml_forecaster.cross_validate`.
    metrics : callable or list of callable.
        One or more metric functions accepted by `ml_forecaster.cross_validate` (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously.
    lags_to_consider : int, optional
        Consider lags `1, 2, ..., lags_to_consider` as candidates.  If `None`, lag selection is skipped.
    candidate_features : list of str, optional
        Column names in `df` that are exogenous feature candidates.  The function never modifies this list.  If `None`, exogenous feature selection is skipped.
    transformations : list, optional
        Lag-transform objects to test as candidates (e.g. `[rolling_mean(3, 1), expanding_std(1)]`).  The function never modifies this list.  If `None`, transform selection is skipped.
    starting_lags : list of int, optional
        Lags to include in the initial feature set before the search begins. These are *not* candidates — they are always included.  Must be a list (e.g. `[1]` or `[1, 2, 3]`).
    starting_transforms : list, optional
        Lag-transform objects to include in the initial feature set before the search begins.  Must be a list.
    best_start_score : list of float, optional
        Initial best scores for each metric. If not provided, the function will compute the baseline score using the model with the starting features (if any) before beginning the search.
    verbose : bool, default False
        Print a message each time a candidate is accepted.
 
    Returns
    -------
    dict
        A dictionary with keys `best_lags`, `best_exogs`, and `best_transforms` containing the selected features.
    """
 
    # ── Normalise metrics to always be a list ─────────────────────────────────
    # cross_validate always expects a list; wrap a lone callable for the user.
    if metrics is None:
        raise ValueError("metrics must be provided (a callable or list of callables).")
    if callable(metrics):
        metrics = [metrics]
 
    # ── step_size default: non-overlapping folds (step = H) ───────────────────
    _step_size = step_size if step_size is not None else H
 
    # ── Validate starting_* arguments ────────────────────────────────────────
    if starting_lags is not None and not isinstance(starting_lags, list):
        raise ValueError("starting_lags must be a list of integers, e.g. [1] or [1, 2, 3].")
    if starting_transforms is not None and not isinstance(starting_transforms, list):
        raise ValueError("starting_transforms must be a list of transformation instances.")
 
    # ── Build candidate pools (local copies — never mutate caller's lists) ────
    remaining_lags = (
        [x for x in range(1, lags_to_consider + 1)
         if x not in (starting_lags or [])]
        if lags_to_consider is not None else []
    )
    remaining_feats = list(candidate_features) if candidate_features is not None else []
    remaining_transforms = list(transformations) if transformations is not None else []
 
    # ── Working df — start without exog candidates, add back as selected ──────
    if candidate_features is not None:
        df_work = df.drop(columns=candidate_features)
        df_orig = df.copy()
    else:
        df_work = df.copy()
        df_orig = df.copy()
 
    # ── Best-feature state ─────────────────────────────────────────────────────
    best_features = {
        "best_lags":       list(starting_lags)      if starting_lags      is not None else [],
        "best_exogs":      [],
        "best_transforms": list(starting_transforms) if starting_transforms is not None else [],
    }
 
    # Current best lags / transforms — kept in sync as candidates are accepted
    current_lags       = list(starting_lags)      if starting_lags      is not None else []
    current_transforms = list(starting_transforms) if starting_transforms is not None else []
 
    best_score = [float('inf')] * len(metrics)
 
    # ── Inner helpers ─────────────────────────────────────────────────────────
 
    def _scores_improved(new_score, ref_score):
        """True only when every metric strictly improves."""
        return all(n < r for n, r in zip(new_score, ref_score))
 
    def _validate(m, df_test):
        """
        Fit-and-score one candidate model via cross-validation.
        Returns a list of mean scores, one per metric.
        """
        result_df, _ = m.cross_validate(
            df=df_test,
            cv_split=cv_split,
            test_size=H,
            metrics=metrics,
            step_size=_step_size,
        )
        # result_df has columns ['eval_metric', 'score']
        return result_df["score"].tolist()
 
    # Columns in candidate_features that are also cat_variables on the model.
    # These must be hidden from the model until they are explicitly selected
    # as exog candidates — otherwise fit() crashes trying to build cat_var
    # for columns that have been dropped from df_work.
    _cat_exog_candidates = set()
    if candidate_features is not None and model.cat_variables is not None:
        _cat_exog_candidates = set(candidate_features) & set(model.cat_variables)
 
    def _make_candidate_model(lags, transforms, active_exogs=None):
        """
        Return a deep copy of the base model configured with the given lags
        and transforms.  active_exogs is the list of exogenous columns
        currently present in the df being passed to cross_validate; any
        cat_variables that are still pending as candidates are stripped from
        the copy so fit() does not look for columns missing from df_work.
        """
        m = model.copy()
        m.n_lag         = sorted(lags)     if lags       else None
        m.lag_transform = list(transforms) if transforms else None
 
        # Remove cat_variables that are still candidate columns not yet added
        if m.cat_variables is not None and _cat_exog_candidates:
            active = set(active_exogs) if active_exogs else set()
            present_cats = [
                c for c in m.cat_variables
                if c not in _cat_exog_candidates or c in active]
            m.cat_variables = present_cats if present_cats else None
 
        return m
 
    # ── Baseline score with starting features (if any) ───────────────────────
    if starting_lags is not None or starting_transforms is not None:
        m_start = _make_candidate_model(current_lags, current_transforms, best_features["best_exogs"])
        best_score = _validate(m_start, df_work)
        if verbose:
            print(f"Baseline score with starting features: {best_score}")
 
    # ── Stepwise forward loop ─────────────────────────────────────────────────
    while True:
        improvement    = False
        best_candidate = {'type': None, 'value': None}

        if best_start_score is not None:
            running_score = best_start_score
        else:
            running_score  = best_score  # tracks the best score seen this iteration
 
        # ── Test lag candidates ───────────────────────────────────────────────
        if remaining_lags:
            for lag in remaining_lags:
                m = _make_candidate_model(current_lags + [lag], current_transforms, best_features["best_exogs"])
                score = _validate(m, df_work)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'lag', 'value': lag}
                    improvement    = True
 
        # ── Test exogenous feature candidates ─────────────────────────────────
        if remaining_feats:
            for feat in remaining_feats:
                df_test = df_work.copy()
                df_test[feat] = df_orig[feat]
                m = _make_candidate_model(current_lags, current_transforms, best_features["best_exogs"] + [feat])
                score = _validate(m, df_test)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'exog', 'value': feat}
                    improvement    = True
 
        # ── Test lag-transform candidates ─────────────────────────────────────
        if remaining_transforms:
            for trans in remaining_transforms:
                m = _make_candidate_model(current_lags, current_transforms + [trans], best_features["best_exogs"])
                score = _validate(m, df_work)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'transform', 'value': trans}
                    improvement    = True
 
        # ── Accept the best candidate found this round ────────────────────────
        if improvement:
            best_score = running_score
            ctype = best_candidate['type']
            cval  = best_candidate['value']
 
            if ctype == 'lag':
                current_lags.append(cval)
                current_lags.sort()
                best_features["best_lags"].append(cval)
                best_features["best_lags"].sort()
                remaining_lags.remove(cval)
 
            elif ctype == 'exog':
                best_features["best_exogs"].append(cval)
                remaining_feats.remove(cval)
                df_work[cval] = df_orig[cval]
 
            elif ctype == 'transform':
                current_transforms.append(cval)
                best_features["best_transforms"].append(cval)
                remaining_transforms.remove(cval)
 
            if verbose:
                label = cval.get_name() if ctype == 'transform' else cval
                print(f"Added {ctype}: {label} | score: {best_score}")
 
        else:
            break  # No candidate improved any metric — search is complete
 
    # ── Finalise output ───────────────────────────────────────────────────────
    # Convert transform objects to their string names for the return value
    best_features["best_transforms"] = [
        t.get_name() for t in best_features["best_transforms"]
    ]
 
    return best_features

In [ ]:
#| echo: false
show_doc(forward_feature_selection)

---

[source](https://github.com/mustafaslanCoto/peshbeen/blob/main/peshbeen/model_selection.py#L318){target="_blank" style="float:right; font-size:smaller"}

### forward_feature_selection

```python

def forward_feature_selection(
    model:object, # A *configured but unfitted* `ml_forecaster` instance.  The function works exclusively on deep copies and never mutates the object passed in.
    df:DataFrame, # Full training DataFrame. Must contain the target column and any candidate exogenous columns.
    cv_split:int, # Number of time-series cross-validation folds.
    H:int, # Forecast horizon (test window size for each fold).
    step_size:Optional=None, # Step size between consecutive CV folds.  If `None` (default) the step equals `H`, producing non-overlapping folds — consistent with the default behaviour of `ml_forecaster.cross_validate`.
    metrics:Union=None, # One or more metric functions accepted by `ml_forecaster.cross_validate` (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously.
    lags_to_consider:Optional=None, # Consider lags `1, 2, ..., lags_to_consider` as candidates.  If `None`, lag selection is skipped.
    candidate_features:Optional=None, # Column names in `df` that are exogenous feature candidates.  The function never modifies this list.  If `None`, exogenous feature selection is skipped.
    transformations:Optional=None, # Lag-transform objects to test as candidates (e.g. `[rolling_mean(3, 1), expanding_std(1)]`).  The function never modifies this list.  If `None`, transform selection is skipped.
    starting_lags:Optional=None, # Lags to include in the initial feature set before the search begins. These are *not* candidates — they are always included.  Must be a list (e.g. `[1]` or `[1, 2, 3]`).
    starting_transforms:Optional=None, # Lag-transform objects to include in the initial feature set before the search begins.  Must be a list.
    best_start_score:Optional=None, # Initial best scores for each metric. If not provided, the function will compute the baseline score using the model with the starting features (if any) before beginning the search.
    verbose:bool=False, # Print a message each time a candidate is accepted.
): # A dictionary with keys `best_lags`, `best_exogs`, and `best_transforms` containing the selected features.


```

*Forward stepwise feature selection for `ml_forecaster` models.*

At each iteration every remaining candidate (lag, exogenous column, or
lag-transform) is tested individually by adding it to the current best
feature set.  The candidate that produces the largest cross-validation
improvement is permanently added.  The loop continues until no remaining
candidate improves any of the evaluation metrics.

In [ ]:
#| echo: false
DocmentTbl(forward_feature_selection)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | object |  | A *configured but unfitted* `ml_forecaster` instance.  The function works exclusively on deep copies and never mutates the object passed in. |
| df | DataFrame |  | Full training DataFrame. Must contain the target column and any candidate exogenous columns. |
| cv_split | int |  | Number of time-series cross-validation folds. |
| H | int |  | Forecast horizon (test window size for each fold). |
| step_size | Optional | None | Step size between consecutive CV folds.  If `None` (default) the step equals `H`, producing non-overlapping folds — consistent with the default behaviour of `ml_forecaster.cross_validate`. |
| metrics | Union | None | One or more metric functions accepted by `ml_forecaster.cross_validate` (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously. |
| lags_to_consider | Optional | None | Consider lags `1, 2, ..., lags_to_consider` as candidates.  If `None`, lag selection is skipped. |
| candidate_features | Optional | None | Column names in `df` that are exogenous feature candidates.  The function never modifies this list.  If `None`, exogenous feature selection is skipped. |
| transformations | Optional | None | Lag-transform objects to test as candidates (e.g. `[rolling_mean(3, 1), expanding_std(1)]`).  The function never modifies this list.  If `None`, transform selection is skipped. |
| starting_lags | Optional | None | Lags to include in the initial feature set before the search begins. These are *not* candidates — they are always included.  Must be a list (e.g. `[1]` or `[1, 2, 3]`). |
| starting_transforms | Optional | None | Lag-transform objects to include in the initial feature set before the search begins.  Must be a list. |
| best_start_score | Optional | None | Initial best scores for each metric. If not provided, the function will compute the baseline score using the model with the starting features (if any) before beginning the search. |
| verbose | bool | False | Print a message each time a candidate is accepted. |
| **Returns** |  |  | **A dictionary with keys `best_lags`, `best_exogs`, and `best_transforms` containing the selected features.** |

In [ ]:
#| hide
ml_linear = ml_forecaster(model=LinearRegression(),
              target_col='admissions', lags = 30, difference=1,
              cat_variables=cat_variables)
feats = forward_feature_selection(
    df=train,
    cv_split=5,
    H=30,
    model=ml_linear,
    metrics=[MAE, RMSE],
    lags_to_consider=7,
    transformations=None,
    starting_lags=None,
    starting_transforms=None,
    verbose=True
)

Added lag: 6 | score: [105.82158083670966, 130.07695764691226]


In [ ]:
#| export

def backward_feature_selection(
    model: object,
    df: pd.DataFrame,
    cv_split: int,
    H: int,
    step_size: Optional[int] = None,
    metrics: Union[Callable, List[Callable]] = None,
    lags_to_consider: Optional[List[int]] = None,
    candidate_features: Optional[List[str]] = None,
    transformations: Optional[List] = None,
    verbose=False,
):
    """
    Backward stepwise feature selection for `ml_forecaster` models.

    Starts with the full feature set (all provided lags, exogenous columns, and
    lag-transforms) and at each iteration tries removing each current feature
    individually.  The feature whose removal produces the largest cross-validation
    improvement is permanently dropped.  The loop continues until no remaining
    feature can be removed without hurting any of the evaluation metrics.

    Parameters
    ----------
    model : ml_forecaster
        A *configured but unfitted* `ml_forecaster` instance.  The function works exclusively on deep copies and never mutates the object passed in.
    df : pd.DataFrame
        Full training DataFrame. Must contain the target column and any candidate exogenous columns.
    cv_split : int
        Number of time-series cross-validation folds.
    H : int
        Forecast horizon (test window size for each fold).
    step_size : int, optional
        Step size between consecutive CV folds.  If `None` (default) the step equals `H`, producing non-overlapping folds.
    metrics : callable or list of callable
        One or more metric functions accepted by `ml_forecaster.cross_validate` (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a feature is only removed when doing so improves **all** metrics simultaneously.
    lags_to_consider : list of int, optional
        Lags to include in the initial feature set and test for removal (e.g. `[1, 2, 3, 4]`).  If `None`, no lag removal is attempted.
    candidate_features : list of str, optional
        Column names in `df` that start in the model and are tested for removal.  If `None`, exogenous feature removal is skipped.
    transformations : list, optional
        Lag-transform objects that start in the model and are tested for removal (e.g. `[rolling_mean(3, 1), expanding_std(1)]`).  If `None`, transform removal is skipped.
    verbose : bool, default False
        Print a message each time a feature is removed.

    Returns
    -------
    dict
        A dictionary with keys `best_lags`, `best_exogs`, and `best_transforms` containing the surviving features after backward selection.
    """

    # ── Normalise metrics to always be a list ─────────────────────────────────
    if metrics is None:
        raise ValueError("metrics must be provided (a callable or list of callables).")
    if callable(metrics):
        metrics = [metrics]

    # ── step_size default: non-overlapping folds (step = H) ───────────────────
    _step_size = step_size if step_size is not None else H

    # ── Build initial feature sets (local copies — never mutate caller's lists) ─
    if lags_to_consider is None:
        current_lags = []
    elif isinstance(lags_to_consider, int):
        current_lags = list(range(1, lags_to_consider + 1))
    else:
        current_lags = sorted(lags_to_consider)
    current_exogs      = list(candidate_features)  if candidate_features is not None else []
    current_transforms = list(transformations)      if transformations    is not None else []


    # ── Working df includes all exog candidates from the start ────────────────
    df_work = df.copy()

    # ── Columns in candidate_features that are also cat_variables on the model ─
    _cat_exog_candidates = set()
    if candidate_features is not None and model.cat_variables is not None:
        _cat_exog_candidates = set(candidate_features) & set(model.cat_variables)

    # ── Inner helpers ─────────────────────────────────────────────────────────

    def _scores_improved(new_score, ref_score):
        """True only when every metric strictly improves."""
        return all(n < r for n, r in zip(new_score, ref_score))

    def _make_candidate_model(lags, transforms, active_exogs=None):
        """
        Return a deep copy of the base model configured with the given lags and
        transforms.  Cat_variables that are no longer in active_exogs are stripped
        from the copy so fit() does not look for missing columns.
        """
        m = model.copy()
        m.n_lag         = sorted(lags)     if lags       else None
        m.lag_transform = list(transforms) if transforms else None

        if m.cat_variables is not None and _cat_exog_candidates:
            active = set(active_exogs) if active_exogs else set()
            present_cats = [
                c for c in m.cat_variables
                if c not in _cat_exog_candidates or c in active]
            m.cat_variables = present_cats if present_cats else None

        return m

    def _validate(m, df_test):
        """Fit-and-score one candidate model via cross-validation."""
        result_df, _ = m.cross_validate(
            df=df_test,
            cv_split=cv_split,
            test_size=H,
            metrics=metrics,
            step_size=_step_size,
        )
        return result_df["score"].tolist()

    # ── Baseline score with all features ──────────────────────────────────────
    m_start = _make_candidate_model(current_lags, current_transforms, current_exogs)
    best_score = _validate(m_start, df_work)
    if verbose:
        print(f"Baseline score with all features: {best_score}")

    # ── Stepwise backward loop ────────────────────────────────────────────────
    while True:
        improvement    = False
        best_candidate = {'type': None, 'value': None}
        running_score  = best_score

        # ── Test lag removals ─────────────────────────────────────────────────
        if current_lags:
            for lag in current_lags:
                reduced_lags = [l for l in current_lags if l != lag]
                m = _make_candidate_model(reduced_lags, current_transforms, current_exogs)
                score = _validate(m, df_work)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'lag', 'value': lag}
                    improvement    = True

        # ── Test exogenous feature removals ───────────────────────────────────
        if current_exogs:
            for feat in current_exogs:
                reduced_exogs = [f for f in current_exogs if f != feat]
                df_test = df_work.drop(columns=[feat])
                m = _make_candidate_model(current_lags, current_transforms, reduced_exogs)
                score = _validate(m, df_test)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'exog', 'value': feat}
                    improvement    = True

        # ── Test lag-transform removals ───────────────────────────────────────
        if current_transforms:
            for trans in current_transforms:
                reduced_transforms = [t for t in current_transforms if t is not trans]
                m = _make_candidate_model(current_lags, reduced_transforms, current_exogs)
                score = _validate(m, df_work)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'type': 'transform', 'value': trans}
                    improvement    = True

        # ── Accept the best removal found this round ──────────────────────────
        if improvement:
            best_score = running_score
            ctype = best_candidate['type']
            cval  = best_candidate['value']

            if ctype == 'lag':
                current_lags.remove(cval)
                if verbose:
                    print(f"Removed lag: {cval} | score: {best_score}")

            elif ctype == 'exog':
                current_exogs.remove(cval)
                df_work = df_work.drop(columns=[cval])
                if verbose:
                    print(f"Removed exog: {cval} | score: {best_score}")

            elif ctype == 'transform':
                current_transforms = [t for t in current_transforms if t is not cval]
                if verbose:
                    label = cval.get_name()
                    print(f"Removed transform: {label} | score: {best_score}")

        else:
            break  # No removal improved any metric — search is complete

    # ── Finalise output ───────────────────────────────────────────────────────
    return {
        "best_lags":       current_lags,
        "best_exogs":      current_exogs,
        "best_transforms": [t.get_name() for t in current_transforms],
    }


In [ ]:
#| echo: false
show_doc(backward_feature_selection)

---

[source](https://github.com/mustafaslanCoto/peshbeen/blob/main/peshbeen/model_selection.py#L562){target="_blank" style="float:right; font-size:smaller"}

### backward_feature_selection

```python

def backward_feature_selection(
    model:object, # A *configured but unfitted* `ml_forecaster` instance.  The function works exclusively on deep copies and never mutates the object passed in.
    df:DataFrame, # Full training DataFrame. Must contain the target column and any candidate exogenous columns.
    cv_split:int, # Number of time-series cross-validation folds.
    H:int, # Forecast horizon (test window size for each fold).
    step_size:Optional=None, # Step size between consecutive CV folds.  If `None` (default) the step equals `H`, producing non-overlapping folds.
    metrics:Union=None, # One or more metric functions accepted by `ml_forecaster.cross_validate` (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a feature is only removed when doing so improves **all** metrics simultaneously.
    lags_to_consider:Optional=None, # Lags to include in the initial feature set and test for removal (e.g. `[1, 2, 3, 4]`).  If `None`, no lag removal is attempted.
    candidate_features:Optional=None, # Column names in `df` that start in the model and are tested for removal.  If `None`, exogenous feature removal is skipped.
    transformations:Optional=None, # Lag-transform objects that start in the model and are tested for removal (e.g. `[rolling_mean(3, 1), expanding_std(1)]`).  If `None`, transform removal is skipped.
    verbose:bool=False, # Print a message each time a feature is removed.
): # A dictionary with keys `best_lags`, `best_exogs`, and `best_transforms` containing the surviving features after backward selection.


```

*Backward stepwise feature selection for `ml_forecaster` models.*

Starts with the full feature set (all provided lags, exogenous columns, and
lag-transforms) and at each iteration tries removing each current feature
individually.  The feature whose removal produces the largest cross-validation
improvement is permanently dropped.  The loop continues until no remaining
feature can be removed without hurting any of the evaluation metrics.

In [ ]:
#| echo: false
DocmentTbl(backward_feature_selection)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | object |  | A *configured but unfitted* `ml_forecaster` instance.  The function works exclusively on deep copies and never mutates the object passed in. |
| df | DataFrame |  | Full training DataFrame. Must contain the target column and any candidate exogenous columns. |
| cv_split | int |  | Number of time-series cross-validation folds. |
| H | int |  | Forecast horizon (test window size for each fold). |
| step_size | Optional | None | Step size between consecutive CV folds.  If `None` (default) the step equals `H`, producing non-overlapping folds. |
| metrics | Union | None | One or more metric functions accepted by `ml_forecaster.cross_validate` (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a feature is only removed when doing so improves **all** metrics simultaneously. |
| lags_to_consider | Optional | None | Lags to include in the initial feature set and test for removal (e.g. `[1, 2, 3, 4]`).  If `None`, no lag removal is attempted. |
| candidate_features | Optional | None | Column names in `df` that start in the model and are tested for removal.  If `None`, exogenous feature removal is skipped. |
| transformations | Optional | None | Lag-transform objects that start in the model and are tested for removal (e.g. `[rolling_mean(3, 1), expanding_std(1)]`).  If `None`, transform removal is skipped. |
| verbose | bool | False | Print a message each time a feature is removed. |
| **Returns** |  |  | **A dictionary with keys `best_lags`, `best_exogs`, and `best_transforms` containing the surviving features after backward selection.** |

In [ ]:
#| hide
ml_linear = ml_forecaster(model=LinearRegression(),
              target_col='admissions', lags = 30, difference=1,
              cat_variables=cat_variables)
feats = backward_feature_selection(
    df=train,
    cv_split=5,
    H=30,
    model=ml_linear,
    metrics=[MAE, RMSE],
    lags_to_consider=15,
    candidate_features=["month"],
    transformations=None,
    verbose=True
)

Baseline score with all features: [115.07834515780135, 137.49870828679713]
Removed lag: 4 | score: [110.96237974796631, 133.769949172414]
Removed lag: 3 | score: [108.1137908619557, 130.98285972051121]
Removed lag: 7 | score: [106.09035226173887, 127.91250425199146]
Removed lag: 5 | score: [104.72210278850693, 126.3568280210822]
Removed lag: 15 | score: [101.99176594446325, 124.98105737252703]
Removed lag: 14 | score: [101.38993154188157, 124.55643414273672]
Removed lag: 13 | score: [101.22117918545669, 124.25819084811606]


## Feature selection methods for multivariate time series models

In [ ]:
#| export

def mv_forward_feature_selection(
    model: object,
    df: pd.DataFrame,
    target_col: str,
    cv_split: int,
    H: int,
    step_size=None,
    metrics=None,
    lags_to_consider=None,
    candidate_features=None,
    transformations=None,
    starting_lags=None,
    starting_transforms=None,
    verbose=False,
):
    """
    Forward stepwise feature selection for `ml_mv_forecaster`.

    Parameters
    ----------
    model : ml_mv_forecaster
        Template model — never mutated.
    df : pd.DataFrame
        DataFrame containing the target variable and any candidate features.
    target_col : str
        Target variable used to evaluate cross-validation score.
    cv_split : int
        Number of time-series cross-validation folds.
    H : int
        Forecast horizon / test size per fold.
    step_size : int, optional
        Rolling-window step size (defaults to H).
    metrics : callable or list of callable
        One or more metric functions (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously.
    lags_to_consider : dict, optional
        ``{col: max_lag}`` — lags 1..max_lag are candidates.
    candidate_features : list of str, optional
        Exogenous columns to consider adding.
    transformations : dict, optional
        ``{col: [transform_objects]}`` — transform candidates per target.
    starting_lags : dict, optional
        Lags already included before search begins.
    starting_transforms : dict, optional
        Transforms already included before search begins.
    verbose : bool, default False

    Returns
    -------
    dict
        `{"best_lags": {col: [...]}, "best_exogs": [...],
           "best_transforms": {col: [name_str, ...]}}`
    """
    if metrics is None:
        raise ValueError("metrics must be provided.")
    if callable(metrics):
        metrics = [metrics]

    _step_size = step_size if step_size is not None else H

    if lags_to_consider is None:
        lags_to_consider = {}
    if transformations is None:
        transformations = {}

    remaining_lags = {col: list(range(1, ml + 1)) for col, ml in lags_to_consider.items()}
    remaining_transforms = {col: list(tl) for col, tl in transformations.items()}
    remaining_feats = list(candidate_features) if candidate_features is not None else []

    best_lags = {col: [] for col in lags_to_consider}
    if starting_lags is not None:
        for col, lags in starting_lags.items():
            best_lags[col] = list(lags)
            remaining_lags[col] = [x for x in remaining_lags.get(col, []) if x not in lags]

    best_transforms = {col: [] for col in transformations}
    if starting_transforms is not None:
        for col, tl in starting_transforms.items():
            best_transforms[col] = list(tl)
            remaining_transforms[col] = [t for t in remaining_transforms.get(col, []) if t not in tl]

    best_features = {
        "best_lags":       best_lags,
        "best_exogs":      [],
        "best_transforms": best_transforms,
    }

    if candidate_features is not None:
        df_work = df.drop(columns=candidate_features)
        df_orig = df.copy()
    else:
        df_work = df.copy()
        df_orig = df.copy()

    best_score = [float('inf')] * len(metrics)

    _cat_exog_candidates = set()
    if candidate_features is not None and model.cat_variables is not None:
        _cat_exog_candidates = set(candidate_features) & set(model.cat_variables)

    def _scores_improved(new_score, ref_score):
        return all(n < r for n, r in zip(new_score, ref_score))

    def _validate(m, df_test):
        # cross_validate returns (metrics_df, cv_df_); metrics_df has columns
        # ["eval_metric", "score"] — one row per metric.
        result_df, _ = m.cross_validate(
            df=df_test, target_col=target_col, cv_split=cv_split,
            test_size=H, metrics=metrics, step_size=_step_size,
        )
        return result_df['score'].tolist()

    def _make_candidate_model(lags_dict, transforms_dict, active_exogs=None):
        m = model.copy()
        # Use {} (not None) when no lags — ml_mv_forecaster.data_prep iterates m.n_lag
        m.n_lag = {col: sorted(v) for col, v in lags_dict.items() if v}
        at = {col: list(v) for col, v in transforms_dict.items() if v}
        m.lag_transform = at if at else None
        if m.cat_variables is not None and _cat_exog_candidates:
            active = set(active_exogs) if active_exogs else set()
            present = [c for c in m.cat_variables
                       if c not in _cat_exog_candidates or c in active]
            m.cat_variables = present if present else None
        return m

    # Evaluate starting point if warm-start provided
    if starting_lags is not None or starting_transforms is not None:
        m_start = _make_candidate_model(
            best_features["best_lags"],
            best_features["best_transforms"],
            best_features["best_exogs"],
        )
        best_score = _validate(m_start, df_work)
        if verbose:
            print(f"Baseline score: {best_score}")

    while True:
        improvement    = False
        best_candidate = {'target': None, 'type': None, 'value': None}
        running_score  = best_score

        # --- try adding each candidate lag ---
        for col, lags in remaining_lags.items():
            for lg in lags:
                trial_lags = {c: list(v) for c, v in best_features["best_lags"].items()}
                trial_lags[col] = sorted(trial_lags[col] + [lg])
                m = _make_candidate_model(trial_lags, best_features["best_transforms"],
                                          best_features["best_exogs"])
                score = _validate(m, df_work)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'target': col, 'type': 'lag', 'value': lg}
                    improvement    = True

        # --- try adding each candidate exog feature ---
        for feat in remaining_feats:
            df_test = df_work.copy()
            df_test[feat] = df_orig[feat]
            active_exogs = best_features["best_exogs"] + [feat]
            m = _make_candidate_model(best_features["best_lags"],
                                      best_features["best_transforms"], active_exogs)
            score = _validate(m, df_test)
            if _scores_improved(score, running_score):
                running_score  = score
                best_candidate = {'target': None, 'type': 'exog', 'value': feat}
                improvement    = True

        # --- try adding each candidate transform ---
        for col, tl in remaining_transforms.items():
            for trans in tl:
                trial_trans = {c: list(v) for c, v in best_features["best_transforms"].items()}
                trial_trans[col] = trial_trans[col] + [trans]
                m = _make_candidate_model(best_features["best_lags"], trial_trans,
                                          best_features["best_exogs"])
                score = _validate(m, df_work)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'target': col, 'type': 'transform', 'value': trans}
                    improvement    = True

        if improvement:
            best_score = running_score
            ctype  = best_candidate['type']
            cval   = best_candidate['value']
            ctargt = best_candidate['target']
            if ctype == 'lag':
                best_features["best_lags"][ctargt].append(cval)
                best_features["best_lags"][ctargt].sort()
                remaining_lags[ctargt].remove(cval)
            elif ctype == 'exog':
                best_features["best_exogs"].append(cval)
                remaining_feats.remove(cval)
                df_work[cval] = df_orig[cval]
            elif ctype == 'transform':
                best_features["best_transforms"][ctargt].append(cval)
                remaining_transforms[ctargt].remove(cval)
            if verbose:
                label = cval.get_name() if ctype == 'transform' else cval
                tgt_str = f" [{ctargt}]" if ctargt else ""
                print(f"Added {ctype}{tgt_str}: {label} | score: {best_score}")
        else:
            break

    for col in best_features["best_lags"]:
        best_features["best_lags"][col].sort()
    best_features["best_transforms"] = {
        col: [t.get_name() for t in tl]
        for col, tl in best_features["best_transforms"].items()
    }
    return best_features

In [ ]:
#| echo: false
show_doc(mv_forward_feature_selection)

---

[source](https://github.com/mustafaslanCoto/peshbeen/blob/main/peshbeen/model_selection.py#L757){target="_blank" style="float:right; font-size:smaller"}

### mv_forward_feature_selection

```python

def mv_forward_feature_selection(
    model:object, # Template model — never mutated.
    df:DataFrame, # DataFrame containing the target variable and any candidate features.
    target_col:str, # Target variable used to evaluate cross-validation score.
    cv_split:int, # Number of time-series cross-validation folds.
    H:int, # Forecast horizon / test size per fold.
    step_size:NoneType=None, # Rolling-window step size (defaults to H).
    metrics:NoneType=None, # One or more metric functions (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously.
    lags_to_consider:NoneType=None, # ``{col: max_lag}`` — lags 1..max_lag are candidates.
    candidate_features:NoneType=None, # Exogenous columns to consider adding.
    transformations:NoneType=None, # ``{col: [transform_objects]}`` — transform candidates per target.
    starting_lags:NoneType=None, # Lags already included before search begins.
    starting_transforms:NoneType=None, # Transforms already included before search begins.
    verbose:bool=False
): # `{"best_lags": {col: [...]}, "best_exogs": [...],
   "best_transforms": {col: [name_str, ...]}}`


```

*Forward stepwise feature selection for `ml_mv_forecaster`.*

In [ ]:
#| echo: false
DocmentTbl(mv_forward_feature_selection)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | object |  | Template model — never mutated. |
| df | DataFrame |  | DataFrame containing the target variable and any candidate features. |
| target_col | str |  | Target variable used to evaluate cross-validation score. |
| cv_split | int |  | Number of time-series cross-validation folds. |
| H | int |  | Forecast horizon / test size per fold. |
| step_size | NoneType | None | Rolling-window step size (defaults to H). |
| metrics | NoneType | None | One or more metric functions (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously. |
| lags_to_consider | NoneType | None | ``{col: max_lag}`` — lags 1..max_lag are candidates. |
| candidate_features | NoneType | None | Exogenous columns to consider adding. |
| transformations | NoneType | None | ``{col: [transform_objects]}`` — transform candidates per target. |
| starting_lags | NoneType | None | Lags already included before search begins. |
| starting_transforms | NoneType | None | Transforms already included before search begins. |
| verbose | bool | False |  |
| **Returns** |  |  | **`{"best_lags": {col: [...]}, "best_exogs": [...],
   "best_transforms": {col: [name_str, ...]}}`** |

In [ ]:
#| hide

from peshbeen.datasets import load_admission_calls
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, HistGradientBoostingRegressor
from catboost import CatBoostRegressor
from cubist import Cubist

from peshbeen.models import ml_mv_forecaster


## get day of week and month as features from the date index
load_admission_calls["day_of_week"] = load_admission_calls.index.dayofweek
load_admission_calls["month"] = load_admission_calls.index.month
train = load_admission_calls[:-30]
test = load_admission_calls[-30:]

cat_variables = ["day_of_week", "month"]
ml_linear = ml_mv_forecaster(model=LinearRegression(),
              target_cols=['admissions', "calls"], lags = {"admissions": 7, "calls": 7},
                cat_variables=cat_variables,
                trend={"admissions": "linear", "calls": "linear"}, change_points={"admissions": [100], "calls": [130]}, pol_degree={"admissions": 1, "calls": 1})
# ml_linear.fit(train)
# forecasts = ml_linear.forecast(H=30, exog=test[cat_variables])

In [ ]:
#| hide
feats_mvf = mv_forward_feature_selection(
    model=ml_linear,
    df=train,
    target_col='admissions',
    cv_split=3,
    H=30,
    metrics=[MAE, RMSE],
    lags_to_consider={"admissions": 7, "calls": 7},
    candidate_features=cat_variables,
    verbose=True
)

Added exog: month | score: [166.74875519029283, 215.17550849516036]
Added lag [admissions]: 1 | score: [128.6669743160086, 159.75541150704532]
Added exog: day_of_week | score: [110.82326327399437, 141.59466251843367]
Added lag [calls]: 1 | score: [103.28304873208835, 133.95230334295886]
Added lag [calls]: 2 | score: [102.08224996482687, 132.4844287010094]


In [ ]:
#| export

def mv_backward_feature_selection(
    model: object,
    df: pd.DataFrame,
    target_col: str,
    cv_split: int,
    H: int,
    step_size=None,
    metrics=None,
    lags_to_consider=None,
    candidate_features=None,
    transformations=None,
    verbose=False,
):
    """
    Backward stepwise feature selection for ``ml_mv_forecaster``.

    Starts with all candidate features included and iteratively removes
    the one whose removal most improves cross-validation score.

    Parameters
    ----------
    model : ml_mv_forecaster
        Template model — never mutated.
    df : pd.DataFrame
        All candidate exog columns must already be present.
    target_col : str
        Target variable used to evaluate cross-validation score.
    cv_split : int
    H : int
        Forecast horizon / test size per fold.
    step_size : int, optional
        Rolling-window step size (defaults to H).
    metrics : callable or list of callable
        One or more metric functions (e.g. `[MAE, RMSE]`). A feature is only removed when its removal improves **all** metrics simultaneously.
    lags_to_consider : dict, optional
        ``{col: max_lag}`` — all lags 1..max_lag start as selected.
    candidate_features : list of str, optional
        Exogenous columns that start as selected.
    transformations : dict, optional
        ``{col: [transform_objects]}`` — all transforms start as selected.
    verbose : bool, default False

    Returns
    -------
    dict
        ``{"best_lags": {col: [...]}, "best_exogs": [...],
           "best_transforms": {col: [name_str, ...]}}``
    """
    if metrics is None:
        raise ValueError("metrics must be provided.")
    if callable(metrics):
        metrics = [metrics]

    _step_size = step_size if step_size is not None else H

    best_features = {
        "best_lags": {col: list(range(1, ml + 1))
                      for col, ml in (lags_to_consider or {}).items()},
        "best_exogs": list(candidate_features) if candidate_features is not None else [],
        "best_transforms": {col: list(tl) for col, tl in (transformations or {}).items()},
    }

    df_work = df.copy()

    _cat_exog_candidates = set()
    if candidate_features is not None and model.cat_variables is not None:
        _cat_exog_candidates = set(candidate_features) & set(model.cat_variables)

    def _scores_improved(new_score, ref_score):
        return all(n < r for n, r in zip(new_score, ref_score))

    def _validate(m, df_test):
        result_df, _ = m.cross_validate(
            df=df_test, target_col=target_col, cv_split=cv_split,
            test_size=H, metrics=metrics, step_size=_step_size,
        )
        return result_df['score'].tolist()

    def _make_candidate_model(lags_dict, transforms_dict, active_exogs=None):
        m = model.copy()
        m.n_lag = {col: sorted(v) for col, v in lags_dict.items() if v}
        at = {col: list(v) for col, v in transforms_dict.items() if v}
        m.lag_transform = at if at else None
        if m.cat_variables is not None and _cat_exog_candidates:
            active = set(active_exogs) if active_exogs else set()
            present = [c for c in m.cat_variables
                       if c not in _cat_exog_candidates or c in active]
            m.cat_variables = present if present else None
        return m

    # Evaluate baseline (all features included) before starting removal
    m_baseline = _make_candidate_model(
        best_features["best_lags"],
        best_features["best_transforms"],
        best_features["best_exogs"],
    )
    best_score = _validate(m_baseline, df_work)
    if verbose:
        print(f"Baseline score (all features): {best_score}")

    while True:
        improvement    = False
        best_candidate = {'target': None, 'type': None, 'value': None}
        running_score  = best_score

        # --- try removing each selected lag ---
        for targ_col, lags in best_features["best_lags"].items():
            for lg in lags:
                trial_lags = {col: list(v) for col, v in best_features["best_lags"].items()}
                trial_lags[targ_col] = sorted([x for x in lags if x != lg])
                m = _make_candidate_model(trial_lags, best_features["best_transforms"],
                                          best_features["best_exogs"])
                score = _validate(m, df_work)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'target': targ_col, 'type': 'lag', 'value': lg}
                    improvement    = True

        # --- try removing each selected transform ---
        for targ_col, tl in best_features["best_transforms"].items():
            for tr in tl:
                trial_trans = {col: list(v) for col, v in best_features["best_transforms"].items()}
                trial_trans[targ_col] = [x for x in tl if x is not tr]
                m = _make_candidate_model(best_features["best_lags"], trial_trans,
                                          best_features["best_exogs"])
                score = _validate(m, df_work)
                if _scores_improved(score, running_score):
                    running_score  = score
                    best_candidate = {'target': targ_col, 'type': 'transform', 'value': tr}
                    improvement    = True

        # --- try removing each selected exog feature ---
        for feat in best_features["best_exogs"]:
            df_test      = df_work.drop(columns=[feat])
            active_after = [x for x in best_features["best_exogs"] if x != feat]
            m = _make_candidate_model(best_features["best_lags"],
                                      best_features["best_transforms"], active_after)
            score = _validate(m, df_test)
            if _scores_improved(score, running_score):
                running_score  = score
                best_candidate = {'target': None, 'type': 'exog', 'value': feat}
                improvement    = True

        if improvement:
            best_score = running_score
            ctype  = best_candidate['type']
            cval   = best_candidate['value']
            ctargt = best_candidate['target']
            if ctype == 'lag':
                best_features["best_lags"][ctargt].remove(cval)
            elif ctype == 'exog':
                best_features["best_exogs"].remove(cval)
                df_work = df_work.drop(columns=[cval])
            elif ctype == 'transform':
                best_features["best_transforms"][ctargt].remove(cval)
            if verbose:
                label = cval.get_name() if ctype == 'transform' else cval
                tgt_str = f" [{ctargt}]" if ctargt else ""
                print(f"Removed {ctype}{tgt_str}: {label} | score: {best_score}")
        else:
            break

    for col in best_features["best_lags"]:
        best_features["best_lags"][col].sort()
    best_features["best_transforms"] = {
        col: [t.get_name() for t in tl]
        for col, tl in best_features["best_transforms"].items()
    }
    return best_features

In [ ]:
#| echo: false
show_doc(mv_backward_feature_selection)

---

[source](https://github.com/mustafaslanCoto/peshbeen/blob/main/peshbeen/model_selection.py#L968){target="_blank" style="float:right; font-size:smaller"}

### mv_backward_feature_selection

```python

def mv_backward_feature_selection(
    model:object, # Template model — never mutated.
    df:DataFrame, # All candidate exog columns must already be present.
    target_col:str, # Target variable used to evaluate cross-validation score.
    cv_split:int, H:int, # Forecast horizon / test size per fold.
    step_size:NoneType=None, # Rolling-window step size (defaults to H).
    metrics:NoneType=None, # One or more metric functions (e.g. `[MAE, RMSE]`). A feature is only removed when its removal improves **all** metrics simultaneously.
    lags_to_consider:NoneType=None, # ``{col: max_lag}`` — all lags 1..max_lag start as selected.
    candidate_features:NoneType=None, # Exogenous columns that start as selected.
    transformations:NoneType=None, # ``{col: [transform_objects]}`` — all transforms start as selected.
    verbose:bool=False
): # ``{"best_lags": {col: [...]}, "best_exogs": [...],
   "best_transforms": {col: [name_str, ...]}}``


```

*Backward stepwise feature selection for ``ml_mv_forecaster``.*

Starts with all candidate features included and iteratively removes
the one whose removal most improves cross-validation score.

In [ ]:
#| echo: false
DocmentTbl(mv_backward_feature_selection)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | object |  | Template model — never mutated. |
| df | DataFrame |  | All candidate exog columns must already be present. |
| target_col | str |  | Target variable used to evaluate cross-validation score. |
| cv_split | int |  |  |
| H | int |  | Forecast horizon / test size per fold. |
| step_size | NoneType | None | Rolling-window step size (defaults to H). |
| metrics | NoneType | None | One or more metric functions (e.g. `[MAE, RMSE]`). A feature is only removed when its removal improves **all** metrics simultaneously. |
| lags_to_consider | NoneType | None | ``{col: max_lag}`` — all lags 1..max_lag start as selected. |
| candidate_features | NoneType | None | Exogenous columns that start as selected. |
| transformations | NoneType | None | ``{col: [transform_objects]}`` — all transforms start as selected. |
| verbose | bool | False |  |
| **Returns** |  |  | **``{"best_lags": {col: [...]}, "best_exogs": [...],
   "best_transforms": {col: [name_str, ...]}}``** |

In [ ]:
#| hide
feats_mvb = mv_backward_feature_selection(
    model=ml_linear,
    df=train,
    target_col='admissions',
    cv_split=3,
    H=30,
    metrics=[MAE, RMSE],
    lags_to_consider={"admissions": 7, "calls": 7},
    candidate_features=cat_variables,
    verbose=True
)

Baseline score (all features): [109.29717677966882, 148.2983103786052]
Removed lag [admissions]: 1 | score: [105.91419788677241, 146.7308283922569]
Removed lag [admissions]: 7 | score: [104.52863353888797, 142.8577988679913]
Removed lag [calls]: 4 | score: [103.42189903530193, 141.93740105887397]
Removed lag [calls]: 7 | score: [102.46152777327569, 140.19391221941422]
Removed lag [calls]: 6 | score: [101.56647138611778, 139.3055534989342]
Removed lag [calls]: 5 | score: [100.81916910153011, 138.00523059549755]
Removed lag [admissions]: 6 | score: [100.78807102377961, 137.00717747150503]
Removed lag [admissions]: 4 | score: [100.6223960783894, 136.9978614491794]


In [ ]:
#| hide

from peshbeen.models import var
var_model = var(target_cols=['admissions', "calls"], lags={'admissions': 7, "calls": 7}, trend={'admissions': "linear", "calls": "linear"},
                cat_variables=cat_variables, change_points={'admissions': [100], "calls": [130]})
feats_mvf_var = mv_forward_feature_selection(
    model=var_model,
    df=train,
    target_col='admissions',
    cv_split=3,
    H=30,
    metrics=[MAE, RMSE],
    lags_to_consider={"admissions": 7, "calls": 7},
    candidate_features=cat_variables,
    verbose=True
)
feats_mvf_var

Added exog: month | score: [166.7487551902926, 215.1755084951603]
Added lag [admissions]: 1 | score: [128.66697431600824, 159.75541150704555]
Added exog: day_of_week | score: [110.82326327395697, 141.5946625184409]
Added lag [calls]: 1 | score: [103.28304873207686, 133.95230334297227]
Added lag [calls]: 2 | score: [102.08224996488683, 132.48442870107655]


{'best_lags': {'admissions': [1], 'calls': [1, 2]},
 'best_exogs': ['month', 'day_of_week'],
 'best_transforms': {}}

In [ ]:
#| hide
feats_mvb_var = mv_backward_feature_selection(
    model=var_model,
    df=train,
    target_col='admissions',
    cv_split=3,
    H=30,
    metrics=[MAE, RMSE],
    lags_to_consider={"admissions": 15, "calls": 15},
    verbose=True
)
feats_mvb_var

Baseline score (all features): [153.61836133409375, 200.96962388047757]
Removed lag [calls]: 13 | score: [148.40591785650653, 195.15154160181564]
Removed lag [calls]: 14 | score: [146.53622692809975, 190.33325164101313]
Removed lag [admissions]: 9 | score: [145.32632489604384, 189.13277347187727]
Removed lag [admissions]: 8 | score: [143.99157468794326, 187.864887356342]
Removed lag [admissions]: 6 | score: [143.00217048719267, 186.88430243234293]
Removed lag [admissions]: 5 | score: [141.47111313514355, 186.14884612640682]
Removed lag [calls]: 7 | score: [140.37455534939866, 185.25906199527356]
Removed lag [calls]: 5 | score: [139.56782762739422, 184.726316487021]
Removed lag [calls]: 10 | score: [138.7648250731925, 183.4532103936001]
Removed lag [admissions]: 14 | score: [138.63006459726742, 183.41669701370458]


{'best_lags': {'admissions': [1, 2, 3, 4, 7, 10, 11, 12, 13, 15],
  'calls': [1, 2, 3, 4, 6, 8, 9, 11, 12, 15]},
 'best_exogs': [],
 'best_transforms': {}}

## Feature selection methods for Markov Switching Autoregressive Regression

In [ ]:
#| export

def ms_arr_forward_feature_selection(
    model: object,
    df: pd.DataFrame,
    cv_split: int,
    H: int,
    step_size: Optional[int] = None,
    metrics: Union[Callable, List[Callable]] = None,
    lags_to_consider: Optional[Union[int, List[int]]] = None,
    candidate_features: Optional[List[str]] = None,
    transformations: Optional[List] = None,
    starting_lags: Optional[List[int]] = None,
    starting_transforms: Optional[List] = None,
    validation_type: str = "cv",
    iterations: int = 10,
    verbose: bool = False,
):
    """
    Forward stepwise feature selection for `ms_arr` models.

    At each iteration every remaining candidate (lag, exogenous column, or
    lag-transform) is tested individually by adding it to the current best
    feature set.  The candidate that produces the largest improvement is
    permanently added.  The loop continues until no remaining candidate
    improves the evaluation criterion.

    The HMM state (A, pi, stds, coeffs) is warm-started from the round
    winner and propagated to subsequent rounds for consistent initialisation.

    Parameters
    ----------
    model : ms_arr
        A configured `ms_arr` instance with `fit_em()` already called (recommended to use few EM iterations for this initial fit, e.g. `iterations=10`) or a template model with the same configuration but not yet fitted.  The model is copied internally and never mutated, so the caller's instance remains unchanged.
    df : pd.DataFrame
        Full training DataFrame. Must contain the target column and any candidate exogenous columns.
    cv_split : int
        Number of time-series cross-validation folds.
    H : int
        Forecast horizon (test window size for each fold).
    step_size : int, optional
        Step size between consecutive CV folds. Defaults to H.
    metrics : callable or list of callable
        Required when validation_type='cv'. Selection driven by first metric; a candidate is accepted only when it improves all metrics.
    lags_to_consider : int or list of int, optional
        Candidate lags. Int → 1..n; list → specific lags.
    candidate_features : list of str, optional
        Exogenous column names to test as candidates.
    transformations : list, optional
        Lag-transform objects to test as candidates.
    starting_lags : list of int, optional
        Lags always included in the initial set (not candidates).
    starting_transforms : list, optional
        Transforms always included in the initial set (not candidates).
    validation_type : str, default 'cv'
        Criterion for selection: 'cv', 'AIC', 'BIC', or 'AIC_BIC'. When 'cv', metrics must be provided and drive selection. When 'AIC' or 'BIC', the respective information criterion is used. When 'AIC_BIC', a candidate is accepted only if it improves both AIC and BIC.
    iterations : int, default 100
        EM iterations used inside fit_em() for each candidate evaluation.
    verbose : bool, default False
        Print a message each time a candidate is accepted.

    Returns
    -------
    dict
         `{"best_lags": [...], "best_exogs": [...], "best_transforms": [...]}`

    """

    if metrics is not None and callable(metrics):
        metrics = [metrics]
    if validation_type == "cv" and metrics is None:
        raise ValueError("metrics must be provided when validation_type='cv'.")
    if starting_lags is not None and not isinstance(starting_lags, list):
        raise ValueError("starting_lags must be a list of integers.")
    if starting_transforms is not None and not isinstance(starting_transforms, list):
        raise ValueError("starting_transforms must be a list of transform instances.")

    _step_size = step_size if step_size is not None else H

    # ── Local working copy — caller's model NEVER touched ─────────────────────
    local_model = model.copy()

    # ── Candidate pools ───────────────────────────────────────────────────────
    if lags_to_consider is not None:
        _all = (list(range(1, lags_to_consider + 1))
                if isinstance(lags_to_consider, int) else list(lags_to_consider))
        remaining_lags = [x for x in _all if x not in (starting_lags or [])]
    else:
        remaining_lags = []

    remaining_feats      = list(candidate_features) if candidate_features is not None else []
    remaining_transforms = list(transformations)    if transformations    is not None else []

    # ── Apply starting features to local_model ────────────────────────────────
    local_model.lags          = list(starting_lags)       if starting_lags       is not None else None
    local_model.lag_transform = list(starting_transforms) if starting_transforms is not None else None

    # ── Working df ────────────────────────────────────────────────────────────
    if candidate_features is not None:
        df_work = df.drop(columns=candidate_features)
        df_orig = df.copy()
    else:
        df_work = df.copy()
        df_orig = df.copy()

    best_features = {
        "best_lags":       list(starting_lags)       if starting_lags       is not None else [],
        "best_exogs":      [],
        "best_transforms": list(starting_transforms) if starting_transforms is not None else [],
    }

    # ── Initial best score ────────────────────────────────────────────────────
    if validation_type == "cv":
        best_score = [float('inf')] * len(metrics)
    elif validation_type in ("AIC", "BIC"):
        best_score = float('inf')
    else:  # AIC_BIC
        best_score = [float('inf'), float('inf')]

    # ── Cat variable handling ─────────────────────────────────────────────────
    _cat_exog_candidates = set()
    if candidate_features is not None and model.cat_variables is not None:
        _cat_exog_candidates = set(candidate_features) & set(model.cat_variables)

    def _strip_pending_cats(m, active_exogs):
        if m.cat_variables is not None and _cat_exog_candidates:
            active = set(active_exogs) if active_exogs else set()
            present = [c for c in m.cat_variables
                       if c not in _cat_exog_candidates or c in active]
            m.cat_variables = present if present else None

    # ── Inner helpers ─────────────────────────────────────────────────────────

    def _scores_improved(new_score, ref_score):
        if isinstance(ref_score, list):
            return all(n < r for n, r in zip(new_score, ref_score))
        return new_score < ref_score

    def _train(m, df_test):
        """Full EM fit — resets coeffs so data_prep reinitialises for new feature set."""
        m.coeffs = None  # shape changes with each candidate — must reinitialise
        m.iter = iterations
        m.fit_em(df_test)
        return m

    def _validate(m, df_test):
        if validation_type == "cv":
            result, _ = m.cross_validate(
                df=df_test, cv_split=cv_split, test_size=H,
                metrics=metrics, step_size=_step_size, n_iter=iterations,
            )
            return result["score"].tolist()
        elif validation_type == "BIC":
            return m.bic
        elif validation_type == "AIC":
            return m.aic
        else:  # AIC_BIC
            return [m.aic, m.bic]

    def _propagate_hmm_state(src, dst):
        """Copy fitted HMM state from src → dst for warm-starting next round."""
        dst.A      = src.A.copy()
        dst.pi     = src.pi.copy()
        dst.stds   = src.stds.copy()
        # dst.coeffs = src.coeffs.copy()

    # ── Baseline score with starting features ────────────────────────────────
    if starting_lags is not None or starting_transforms is not None:
        m_start = local_model.copy()
        _strip_pending_cats(m_start, best_features["best_exogs"])
        _train(m_start, df_work)
        best_score = _validate(m_start, df_work)
        _propagate_hmm_state(m_start, local_model)
        if verbose:
            print(f"Baseline score: {best_score}")

    # ── Stepwise forward loop ─────────────────────────────────────────────────
    while True:
        improvement    = False
        best_candidate = {'type': None, 'value': None, 'model': None}
        running_score  = best_score

        # ── Lag candidates ────────────────────────────────────────────────────
        for lag in remaining_lags:
            m = local_model.copy()
            m.lags = sorted(best_features["best_lags"] + [lag])
            _strip_pending_cats(m, best_features["best_exogs"])
            _train(m, df_work)
            score = _validate(m, df_work)
            if _scores_improved(score, running_score):
                running_score  = score
                best_candidate = {'type': 'lag', 'value': lag, 'model': m}
                improvement    = True

        # ── Exogenous feature candidates ──────────────────────────────────────
        for feat in remaining_feats:
            df_test = df_work.copy()
            df_test[feat] = df_orig[feat]
            active_exogs = best_features["best_exogs"] + [feat]
            m = local_model.copy()
            _strip_pending_cats(m, active_exogs)
            _train(m, df_test)
            score = _validate(m, df_test)
            if _scores_improved(score, running_score):
                running_score  = score
                best_candidate = {'type': 'exog', 'value': feat, 'model': m}
                improvement    = True

        # ── Transform candidates ──────────────────────────────────────────────
        for trans in remaining_transforms:
            m = local_model.copy()
            m.lag_transform = list(best_features["best_transforms"]) + [trans]
            _strip_pending_cats(m, best_features["best_exogs"])
            _train(m, df_work)
            score = _validate(m, df_work)
            if _scores_improved(score, running_score):
                running_score  = score
                best_candidate = {'type': 'transform', 'value': trans, 'model': m}
                improvement    = True

        # ── Accept best candidate ─────────────────────────────────────────────
        if improvement:
            best_score = running_score
            ctype = best_candidate['type']
            cval  = best_candidate['value']
            m_won = best_candidate['model']

            if ctype == 'lag':
                best_features["best_lags"].append(cval)
                best_features["best_lags"].sort()
                remaining_lags.remove(cval)
                local_model.lags = list(best_features["best_lags"])

            elif ctype == 'exog':
                best_features["best_exogs"].append(cval)
                remaining_feats.remove(cval)
                df_work[cval] = df_orig[cval]

            elif ctype == 'transform':
                best_features["best_transforms"].append(cval)
                remaining_transforms.remove(cval)
                local_model.lag_transform = list(best_features["best_transforms"])

            _propagate_hmm_state(m_won, local_model)

            if verbose:
                label = cval.get_name() if ctype == 'transform' else cval
                print(f"Added {ctype}: {label} | score: {best_score}")
        else:
            break

    # ── Finalise — refit on full df_work with selected features ───────────────
    model_ = local_model.copy()
    model_.lags          = list(best_features["best_lags"])       if best_features["best_lags"]       else None
    model_.lag_transform = list(best_features["best_transforms"]) if best_features["best_transforms"] else None
    _strip_pending_cats(model_, best_features["best_exogs"])  # ← add this
    _train(model_, df_work)

    best_features["best_transforms"] = [t.get_name() for t in best_features["best_transforms"]]
    return best_features, model_

In [ ]:
#| echo: false
show_doc(mv_forward_feature_selection)

---

[source](https://github.com/mustafaslanCoto/peshbeen/blob/main/peshbeen/model_selection.py#L757){target="_blank" style="float:right; font-size:smaller"}

### mv_forward_feature_selection

```python

def mv_forward_feature_selection(
    model:object, # Template model — never mutated.
    df:DataFrame, # DataFrame containing the target variable and any candidate features.
    target_col:str, # Target variable used to evaluate cross-validation score.
    cv_split:int, # Number of time-series cross-validation folds.
    H:int, # Forecast horizon / test size per fold.
    step_size:NoneType=None, # Rolling-window step size (defaults to H).
    metrics:NoneType=None, # One or more metric functions (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously.
    lags_to_consider:NoneType=None, # ``{col: max_lag}`` — lags 1..max_lag are candidates.
    candidate_features:NoneType=None, # Exogenous columns to consider adding.
    transformations:NoneType=None, # ``{col: [transform_objects]}`` — transform candidates per target.
    starting_lags:NoneType=None, # Lags already included before search begins.
    starting_transforms:NoneType=None, # Transforms already included before search begins.
    verbose:bool=False
): # `{"best_lags": {col: [...]}, "best_exogs": [...],
   "best_transforms": {col: [name_str, ...]}}`


```

*Forward stepwise feature selection for `ml_mv_forecaster`.*

In [ ]:
#| echo: false
DocmentTbl(mv_forward_feature_selection)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | object |  | Template model — never mutated. |
| df | DataFrame |  | DataFrame containing the target variable and any candidate features. |
| target_col | str |  | Target variable used to evaluate cross-validation score. |
| cv_split | int |  | Number of time-series cross-validation folds. |
| H | int |  | Forecast horizon / test size per fold. |
| step_size | NoneType | None | Rolling-window step size (defaults to H). |
| metrics | NoneType | None | One or more metric functions (e.g. `[MAE, RMSE]`). Selection is driven by the **first** metric in the list; a candidate is only accepted when it improves **all** metrics simultaneously. |
| lags_to_consider | NoneType | None | ``{col: max_lag}`` — lags 1..max_lag are candidates. |
| candidate_features | NoneType | None | Exogenous columns to consider adding. |
| transformations | NoneType | None | ``{col: [transform_objects]}`` — transform candidates per target. |
| starting_lags | NoneType | None | Lags already included before search begins. |
| starting_transforms | NoneType | None | Transforms already included before search begins. |
| verbose | bool | False |  |
| **Returns** |  |  | **`{"best_lags": {col: [...]}, "best_exogs": [...],
   "best_transforms": {col: [name_str, ...]}}`** |

In [ ]:
#| hide
from peshbeen.datasets import load_wales_admissions
load_wales_admissions["day_of_week"] = load_wales_admissions.index.dayofweek
load_wales_admissions["month"] = load_wales_admissions.index.month
# split the data into train and test sets
train = load_wales_admissions[:-30]
test = load_wales_admissions[-30:]

from peshbeen.models import ms_arr

ms_model = ms_arr(n_components=2, target_col='admissions', lags=14, difference=1,
                  cat_variables=cat_variables, switching_var=True, n_iter=40)

ms_model.fit_em(train)
ms_model.fit(train, n_iter=10)
forecasts = ms_model.forecast(H=30, exog=test[cat_variables])

In [ ]:
#| hide
best_, mod = ms_arr_forward_feature_selection(
    model=ms_model,
    df=train,
    cv_split=3,
    H=30,
    metrics=[MAE, RMSE],
    lags_to_consider=14,
    candidate_features=cat_variables,
    transformations=None,
    starting_lags=None,
    starting_transforms=None,
    validation_type="cv",
    iterations=2,
    verbose=True,
)

Added lag: 1 | score: [157.88817739255657, 199.3932604634297]
Added lag: 13 | score: [156.48683256280455, 197.08971163044995]


In [ ]:
#| export

def ms_arr_backward_feature_selection(
    df: pd.DataFrame,
    cv_split: int,
    H: int,
    step_size: Optional[int] = None,
    model: object = None,
    metrics: Union[Callable, List[Callable]] = None,
    lags_to_consider: Optional[Union[int, List[int]]] = None,
    candidate_features: Optional[List[str]] = None,
    transformations: Optional[List] = None,
    validation_type: str = "cv",
    iterations: int = 100,
    verbose: bool = False,
):
    """
    Backward stepwise feature selection for `ms_arr` models.

    Starts with the full feature set and at each iteration tries removing each
    current feature individually.  The feature whose removal produces the
    largest improvement is permanently dropped.  The loop continues until no
    removal improves the evaluation criterion.

    The HMM state (A, pi, stds, coeffs) is warm-started from the round
    winner and propagated to subsequent rounds.

    Parameters
    ----------
    df : pd.DataFrame
        Full training DataFrame. All candidate exogenous columns must be present.
    cv_split : int
        Number of time-series cross-validation folds.
    H : int
        Forecast horizon (test window size for each fold).
    step_size : int, optional
        Step size between consecutive CV folds. Defaults to H.
    model : ms_arr
        A configured but unfitted ms_arr instance. Never mutated.
    metrics : callable or list of callable
        Required when validation_type='cv'. A feature is only removed when
        doing so improves all metrics simultaneously.
    lags_to_consider : int or list of int, optional
        Initial lag set. Int → 1..n; list → specific lags.
    candidate_features : list of str, optional
        Exogenous columns that start in the model and are tested for removal.
    transformations : list, optional
        Lag-transform objects that start in the model and are tested for removal.
    validation_type : str, default 'cv'
        Criterion for selection: 'cv', 'AIC', 'BIC', or 'AIC_BIC'.
    iterations : int, default 100
        EM iterations used inside fit_em() for each candidate evaluation.
    verbose : bool, default False
        Print a message each time a feature is removed.

    Returns
    -------
    dict
        `{"best_lags": [...], "best_exogs": [...], "best_transforms": [...]}`
    """

    if metrics is not None and callable(metrics):
        metrics = [metrics]
    if validation_type == "cv" and metrics is None:
        raise ValueError("metrics must be provided when validation_type='cv'.")

    _step_size = step_size if step_size is not None else H

    # ── Local working copy ────────────────────────────────────────────────────
    local_model = model.copy()

    # ── Initial feature sets ──────────────────────────────────────────────────
    if lags_to_consider is not None:
        current_lags = (list(range(1, lags_to_consider + 1))
                        if isinstance(lags_to_consider, int) else sorted(lags_to_consider))
    else:
        current_lags = []

    current_exogs      = list(candidate_features) if candidate_features is not None else []
    current_transforms = list(transformations)    if transformations    is not None else []

    local_model.lags          = list(current_lags)       if current_lags       else None
    local_model.lag_transform = list(current_transforms) if current_transforms else None

    df_work = df.copy()

    # ── Cat variable handling ─────────────────────────────────────────────────
    _cat_exog_candidates = set()
    if candidate_features is not None and model.cat_variables is not None:
        _cat_exog_candidates = set(candidate_features) & set(model.cat_variables)

    def _strip_pending_cats(m, active_exogs):
        if m.cat_variables is not None and _cat_exog_candidates:
            active = set(active_exogs) if active_exogs else set()
            present = [c for c in m.cat_variables
                       if c not in _cat_exog_candidates or c in active]
            m.cat_variables = present if present else None

    # ── Inner helpers ─────────────────────────────────────────────────────────

    def _scores_improved(new_score, ref_score):
        if isinstance(ref_score, list):
            return all(n < r for n, r in zip(new_score, ref_score))
        return new_score < ref_score

    # def _train(m, df_test):
    #     m.iter = iterations
    #     m.fit_em(df_test)
    #     return m
    
    def _train(m, df_test):
        """Full EM fit — resets coeffs so data_prep reinitialises for new feature set."""
        m.coeffs = None  # shape changes with each candidate — must reinitialise
        m.iter = iterations
        m.fit_em(df_test)
        return m

    def _validate(m, df_test):
        if validation_type == "cv":
            result, _ = m.cross_validate(
                df=df_test, cv_split=cv_split, test_size=H,
                metrics=metrics, step_size=_step_size, n_iter=iterations,
            )
            return result["score"].tolist()
        elif validation_type == "BIC":
            return m.bic
        elif validation_type == "AIC":
            return m.aic
        else:  # AIC_BIC
            return [m.aic, m.bic]

    def _propagate_hmm_state(src, dst):
        dst.A      = src.A.copy()
        dst.pi     = src.pi.copy()
        dst.stds   = src.stds.copy()
        dst.coeffs = src.coeffs.copy()

    # ── Baseline score with all features ─────────────────────────────────────
    m_start = local_model.copy()
    _strip_pending_cats(m_start, current_exogs)
    _train(m_start, df_work)
    best_score = _validate(m_start, df_work)
    _propagate_hmm_state(m_start, local_model)
    if verbose:
        print(f"Baseline score with all features: {best_score}")

    # ── Stepwise backward loop ────────────────────────────────────────────────
    while True:
        improvement    = False
        best_candidate = {'type': None, 'value': None, 'model': None}
        running_score  = best_score

        # ── Test lag removals ─────────────────────────────────────────────────
        for lag in current_lags:
            reduced_lags = [l for l in current_lags if l != lag]
            m = local_model.copy()
            m.lags = reduced_lags if reduced_lags else None
            _strip_pending_cats(m, current_exogs)
            _train(m, df_work)
            score = _validate(m, df_work)
            if _scores_improved(score, running_score):
                running_score  = score
                best_candidate = {'type': 'lag', 'value': lag, 'model': m}
                improvement    = True

        # ── Test exog removals ────────────────────────────────────────────────
        for feat in current_exogs:
            reduced_exogs = [f for f in current_exogs if f != feat]
            df_test = df_work.drop(columns=[feat])
            m = local_model.copy()
            _strip_pending_cats(m, reduced_exogs)
            _train(m, df_test)
            score = _validate(m, df_test)
            if _scores_improved(score, running_score):
                running_score  = score
                best_candidate = {'type': 'exog', 'value': feat, 'model': m}
                improvement    = True

        # ── Test transform removals ───────────────────────────────────────────
        for trans in current_transforms:
            reduced_transforms = [t for t in current_transforms if t is not trans]
            m = local_model.copy()
            m.lag_transform = reduced_transforms if reduced_transforms else None
            _strip_pending_cats(m, current_exogs)
            _train(m, df_work)
            score = _validate(m, df_work)
            if _scores_improved(score, running_score):
                running_score  = score
                best_candidate = {'type': 'transform', 'value': trans, 'model': m}
                improvement    = True

        # ── Accept best removal ───────────────────────────────────────────────
        if improvement:
            best_score = running_score
            ctype = best_candidate['type']
            cval  = best_candidate['value']
            m_won = best_candidate['model']

            if ctype == 'lag':
                current_lags.remove(cval)
                local_model.lags = current_lags if current_lags else None
                if verbose:
                    print(f"Removed lag: {cval} | score: {best_score}")

            elif ctype == 'exog':
                current_exogs.remove(cval)
                df_work = df_work.drop(columns=[cval])
                if verbose:
                    print(f"Removed exog: {cval} | score: {best_score}")

            elif ctype == 'transform':
                current_transforms = [t for t in current_transforms if t is not cval]
                local_model.lag_transform = current_transforms if current_transforms else None
                if verbose:
                    label = cval.get_name()
                    print(f"Removed transform: {label} | score: {best_score}")

            _propagate_hmm_state(m_won, local_model)

        else:
            break

    # ── Finalise ──────────────────────────────────────────────────────────────
    model_ = local_model.copy()
    model_.lags          = current_lags       if current_lags       else None
    model_.lag_transform = current_transforms if current_transforms else None
    _strip_pending_cats(model_, current_exogs)  # ← add this
    _train(model_, df_work)


    return {
        "best_lags":       current_lags,
        "best_exogs":      current_exogs,
        "best_transforms": [t.get_name() for t in current_transforms],
    }, model_


In [ ]:
#| echo: false
show_doc(mv_backward_feature_selection)

---

[source](https://github.com/mustafaslanCoto/peshbeen/blob/main/peshbeen/model_selection.py#L968){target="_blank" style="float:right; font-size:smaller"}

### mv_backward_feature_selection

```python

def mv_backward_feature_selection(
    model:object, # Template model — never mutated.
    df:DataFrame, # All candidate exog columns must already be present.
    target_col:str, # Target variable used to evaluate cross-validation score.
    cv_split:int, H:int, # Forecast horizon / test size per fold.
    step_size:NoneType=None, # Rolling-window step size (defaults to H).
    metrics:NoneType=None, # One or more metric functions (e.g. `[MAE, RMSE]`). A feature is only removed when its removal improves **all** metrics simultaneously.
    lags_to_consider:NoneType=None, # ``{col: max_lag}`` — all lags 1..max_lag start as selected.
    candidate_features:NoneType=None, # Exogenous columns that start as selected.
    transformations:NoneType=None, # ``{col: [transform_objects]}`` — all transforms start as selected.
    verbose:bool=False
): # ``{"best_lags": {col: [...]}, "best_exogs": [...],
   "best_transforms": {col: [name_str, ...]}}``


```

*Backward stepwise feature selection for ``ml_mv_forecaster``.*

Starts with all candidate features included and iteratively removes
the one whose removal most improves cross-validation score.

In [ ]:
#| echo: false
DocmentTbl(mv_backward_feature_selection)

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| model | object |  | Template model — never mutated. |
| df | DataFrame |  | All candidate exog columns must already be present. |
| target_col | str |  | Target variable used to evaluate cross-validation score. |
| cv_split | int |  |  |
| H | int |  | Forecast horizon / test size per fold. |
| step_size | NoneType | None | Rolling-window step size (defaults to H). |
| metrics | NoneType | None | One or more metric functions (e.g. `[MAE, RMSE]`). A feature is only removed when its removal improves **all** metrics simultaneously. |
| lags_to_consider | NoneType | None | ``{col: max_lag}`` — all lags 1..max_lag start as selected. |
| candidate_features | NoneType | None | Exogenous columns that start as selected. |
| transformations | NoneType | None | ``{col: [transform_objects]}`` — all transforms start as selected. |
| verbose | bool | False |  |
| **Returns** |  |  | **``{"best_lags": {col: [...]}, "best_exogs": [...],
   "best_transforms": {col: [name_str, ...]}}``** |

In [ ]:
#| hide
best_, mod = ms_arr_backward_feature_selection(
    df=train,
    cv_split=3,
    H=30,
    model=ms_model,
    metrics=[MAE, RMSE],
    lags_to_consider=5,
    candidate_features=cat_variables,
    transformations=None,
    validation_type="cv",
    iterations=2,
    verbose=True,
)

Baseline score with all features: [191.30737728808163, 230.35978661289437]
Removed lag: 4 | score: [141.12164680766503, 174.89153900513801]


In [ ]:
#| hide

# from pyexpat import model
# import pandas as pd
# import numpy as np
# from scipy import stats
# from numba import jit
# from scipy.stats import boxcox
# from scipy.special import inv_boxcox
# from sklearn.linear_model import LinearRegression
# from numba import jit
# ##Stationarity Check
# from statsmodels.tsa.stattools import adfuller, kpss
# from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
# from statsmodels.tsa.holtwinters import ExponentialSmoothing
# import matplotlib.pyplot as plt
# from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# from statistics import NormalDist
# import warnings
# warnings.filterwarnings("ignore")
# from statsmodels.tsa.statespace.sarimax import SARIMAX
# from tqdm import tqdm_notebook
# from itertools import product
# from typing import List, Dict, Optional, Callable, Tuple, Any, Union

# # ------------------------------------------------------------------------------
# # Forward Feature Selection for HMM
# # ------------------------------------------------------------------------------

# def hmm_forward_feature_selection(df, n_folds = None, H = None, model = None, metrics = None,
#                                   lags_to_consider = None, candidate_features = None, transformations = None, 
#                                     step_size = None, starting_lags = None, starting_transforms = None,
#                                     validation_type = "cv", iterations = 10, verbose = False):
#     """
#     Performs forward lag/feature/transform selection for Regression models.
#     Parameters:
#         df (pd.DataFrame): DataFrame containing the time series data.
#         n_folds (int, optional): Number of cross-validation folds.
#         H (int, optional): Forecast horizon.
#         model: Model to be used for training and evaluation.
#         metrics (list, optional): List of metrics to evaluate the model. Even one metric, should be provided in a list.
#         lags_to_consider (list, optional): List of lags to consider for feature selection.
#         candidate_features (list, optional): List of candidate exogenous features.
#         transformations (list, optional): List of transformations to apply.
#         step_size (int, optional): Step size for rolling window.
#         starting_lags (list, optional): List of starting lags.
#         starting_transforms (list, optional): List of starting transformations.
#         validation_type (str, optional): Type of validation to use ("cv", "BIC", "AIC" or both "AIC_BIC"). if "AIC_BIC" are both selected, the model will be evaluated using both criteria.
#         iterations (int, optional): Number of iterations for model fitting to update parameters.
#         verbose (bool, optional): Whether to print progress messages.
#     Returns:
#         dict: Dictionary of best features
#     """


#     if lags_to_consider is not None:
#         if isinstance(lags_to_consider, int):
#             remaining_lags = list(range(1, lags_to_consider + 1))
#         elif isinstance(lags_to_consider, list):
#             remaining_lags = lags_to_consider
#         model.lags = None
#         if starting_lags is not None:
#             if not isinstance(starting_lags, list):
#                 raise ValueError("starting_lags should be a list of integers.")
#             model.lags = starting_lags
#             remaining_lags = [x for x in remaining_lags if x not in starting_lags]

#     if candidate_features is not None:
#         df_orig = df.copy() # Keep original for feature add-ba
#         df = df.drop(columns=candidate_features)
#     if transformations is not None:
#         if starting_transforms is not None:
#             if not isinstance(starting_transforms, list):
#                 raise ValueError("starting_transforms should be a list of transformation instances.")
#             model.lag_transform = starting_transforms
#             transformations = [x for x in transformations if x not in starting_transforms]
#         else:
#             model.lag_transform = None
            
#     best_features = {
#     "best_lags": list(starting_lags) if starting_lags is not None else [],
#     "best_exogs": [],
#     "best_transforms": list(starting_transforms) if starting_transforms is not None else []}

#     if validation_type == "cv":
#         if isinstance(metrics, list):
#             best_score = [float('inf')] * len(metrics)
#         else:
#             best_score = float('inf')
#     elif validation_type in ("BIC", "AIC"):
#         best_score = float('inf')
#     elif validation_type == "AIC_BIC":
#         best_score = [float('inf')] * 2
#     else:
#         raise ValueError("Invalid validation_type. Choose from 'cv', 'BIC', 'AIC', or 'AIC_BIC'.")

#     if isinstance(best_score, list):
#         def is_elementwise_improvement(score, best_s):
#             return all(s < b for s, b in zip(score, best_s))
#     else:
#         def is_elementwise_improvement(score, best_s):
#             return score < best_s

# # After each feature selection step iterate model to make sure parameters are updated like transition probabilities and stds
#     def model_update(model_test, df_test, iterations=iterations):
#         model_test.data_prep(df_test) # update data preparation because if new lags to be consistent with coefficients
#         model_test.compute_coeffs() # update model coefficients because of new lags
#         model_test.fit(df_test, n_iter=iterations) # update model parameters like transition probs and stds
#         return model_test
    

#     def validation(model_test, df_test):
#         if validation_type == "cv":
#             cv_result = hmm_cross_validate(model=model_test, df=df_test, cv_split=n_folds, test_size=H,
#                                 metrics=metrics, step_size=step_size, n_iter=iterations)
    
#             if isinstance(metrics, list):
#                 score = cv_result["score"].tolist()
#             else:
#                 score = cv_result["score"].values[0]
#         elif validation_type == "BIC":
#             score = model_test.BIC
#         elif validation_type == "AIC":
#             score = model_test.AIC
#         elif validation_type == "AIC_BIC":
#             score = [model_test.AIC, model_test.BIC]
#         else:
#             raise ValueError("Invalid validation_type. Choose from 'cv', 'BIC', 'AIC', or 'AIC_BIC'.")

#         return score

#     # Initial score with starting features
#     if starting_lags is not None or starting_transforms is not None:
#         model_start = model.copy()
#         model_start = model_update(model_start, df)
#         best_score = validation(model_start, df)

#     while True:
#         improvement = False
#         candidate = {'type': None, 'name': None}
#         scores = best_score

#         # Test Lags
#         if lags_to_consider is not None:
#             for lag in remaining_lags:
#                 current_lags = sorted(best_features["best_lags"] + [lag])
#                 model_test = model.copy()
#                 model_test.lags = current_lags
#                 model_test = model_update(model_test, df)
#                 score = validation(model_test, df)
#                 if is_elementwise_improvement(score, scores):
#                     scores = score
#                     candidate = {'type': 'lag', 'name': lag, 'model': model_test}
#                     improvement = True

#         # Test Exogenous Features
#         if candidate_features is not None:
#             for feat in candidate_features:
#                 df_test = df.copy()
#                 df_test[feat] = df_orig[feat]
#                 model_test = model.copy()
#                 model_test = model_update(model_test, df_test)
#                 score = validation(model_test, df_test)
#                 if is_elementwise_improvement(score, scores):
#                     scores = score
#                     candidate = {'type': 'exog', 'name': feat, 'model': model_test}
#                     improvement = True

#         # Test Transformations
#         if transformations is not None:
#             for trans in transformations:
#                 model_test = model.copy()
#                 lag_transform = (model_test.lag_transform or []) + [trans]
#                 model_test.lag_transform = lag_transform
#                 model_test = model_update(model_test, df)
#                 score = validation(model_test, df)
#                 if is_elementwise_improvement(score, scores):
#                     scores = score
#                     candidate = {'type': 'transform', 'name': trans, 'model': model_test}
#                     improvement = True

#         # Update best features
#         if improvement:
#             best_score = scores
#             if candidate['type'] == 'lag':
#                 best_features["best_lags"].append(candidate['name'])
#                 remaining_lags.remove(candidate['name'])
#             elif candidate['type'] == 'exog':
#                 best_features["best_exogs"].append(candidate['name'])
#                 candidate_features.remove(candidate['name'])
#                 df[candidate['name']] = df_orig[candidate['name']]
#             elif candidate['type'] == 'transform':
#                 best_features["best_transforms"].append(candidate['name'])
#                 transformations.remove(candidate['name'])
#                 if model.lag_transform is None:
#                     model.lag_transform = [candidate['name']]
#                 else:
#                     model.lag_transform.append(candidate['name'])
#             # update transition probs and stds of states

#             model.A = candidate['model'].A
#             model.stds = candidate['model'].stds
#             model.LL = candidate['model'].LL
#             model.pi = candidate['model'].pi

#             if verbose:
#                 print(f"Added {candidate['type']}: {candidate['name']} with score: {best_score} and loglik and BIC: {model.LL}, {model.BIC}")
#         else:
#             break  # No improvement

#     # Finalize model with best features
#     model_ = model.copy()
#     if lags_to_consider is not None and best_features["best_lags"]:
#         model_.lags = best_features["best_lags"]
#     if transformations is not None and best_features["best_transforms"]:
#         model_.lag_transform = best_features["best_transforms"]
#     model_ = model_update(model_, df)


#     if transformations is not None and best_features["best_transforms"]:
#         best_features["best_transforms"] = [trans.get_name() for trans in best_features["best_transforms"]]
    
#     return best_features, model_



# def hmm_backward_feature_selection(df, n_folds = None, H = None, model = None, metrics = None,
#                                   lags_to_consider = None, candidate_features = None, transformations = None, 
#                                     step_size = None, validation_type = "cv", iterations = 10, verbose = False):
#     """
#     Performs backward lag selection for Regression models.
#     Parameters:
#         df (pd.DataFrame): DataFrame containing the time series data.
#         n_folds (int, optional): Number of cross-validation folds.
#         H (int, optional): Forecast horizon.
#         model: Model to be used for training and evaluation.
#         metrics (list, optional): List of metrics to evaluate the model.
#         lags_to_consider (list, optional): List of lags to consider for feature selection.
#         candidate_features (list, optional): List of candidate exogenous features.
#         transformations (list, optional): List of transformations to apply.
#         step_size (int, optional): Step size for rolling window.
#         validation_type (str, optional): Type of validation to use ("cv", "BIC", "AIC" or both "AIC_BIC"). if "AIC_BIC" are both selected, the model will be evaluated using both criteria.
#         iterations (int, optional): Number of iterations for model fitting to update parameters.
#         verbose (bool, optional): Whether to print progress messages.
#     Returns:
#         dict: Dictionary of best features
#     """
#     remaining_lags = list(range(1, lags_to_consider + 1)) if lags_to_consider is not None else []
#     candidate_features = candidate_features.copy() if candidate_features is not None else []
#     transformations = transformations.copy() if transformations is not None else []
#     best_features = {"best_lags": remaining_lags, "best_exogs": candidate_features, "best_transforms": transformations}

#     ## setting the full model
#     # model_full = model.copy()
#     if lags_to_consider is not None:
#         model.lags = remaining_lags
#     if transformations is not None:
#         model.lag_transform = transformations

#     # set validation
#     if validation_type == "cv":
#         if isinstance(metrics, list):
#             best_score = [float('inf')] * len(metrics)
#         else:
#             best_score = float('inf')
#     elif validation_type in ("BIC", "AIC"):
#         best_score = float('inf')
#     elif validation_type == "AIC_BIC":
#         best_score = [float('inf')] * 2
#     else:
#         raise ValueError("Invalid validation_type. Choose from 'cv', 'BIC', 'AIC', or 'AIC_BIC'.")

#     if isinstance(best_score, list):
#         def is_elementwise_improvement(score, best_s):
#             return all(s < b for s, b in zip(score, best_s))
#     else:
#         def is_elementwise_improvement(score, best_s):
#             return score < best_s
        
# # After each feature selection step iterate model to make sure parameters are updated like transition probabilities and stds
#     def model_update(model_test, df_test, iterations=iterations):
#         model_test.data_prep(df_test) # update data preparation because if new lags to be consistent with coefficients
#         model_test.compute_coeffs() # update model coefficients because of new lags
#         model_test.fit(df_test, n_iter=iterations) # update model parameters like transition probs and stds
#         return model_test
    

#     def validation(model_test, df_test):
#         if validation_type == "cv":
#             cv_result = hmm_cross_validate(model=model_test, df=df_test, cv_split=n_folds, test_size=H,
#                                 metrics=metrics, step_size=step_size, n_iter=iterations)

#             if isinstance(metrics, list):
#                 score = cv_result["score"].tolist()
#             else:
#                 score = cv_result["score"].values[0]
#         elif validation_type == "BIC":
#             score = model_test.BIC
#         elif validation_type == "AIC":
#             score = model_test.AIC
#         elif validation_type == "AIC_BIC":
#             score = [model_test.AIC, model_test.BIC]
#         else:
#             raise ValueError("Invalid validation_type. Choose from 'cv', 'BIC', 'AIC', or 'AIC_BIC'.")

#         return score

#     # best_lags = None
#     while True:
#         improvement = False
#         candidate = {'type': None, 'name': None}
#         scores = best_score
#         if best_features["best_lags"]:
#             for lg in best_features["best_lags"]:
#                 lags_to_test = [x for x in best_features["best_lags"] if x != lg]
#                 lags_to_test.sort()
#                 model_test = model.copy()
#                 model_test.lags = lags_to_test
#                 model_test = model_update(model_test, df)
#                 score = validation(model_test, df)
#                 if is_elementwise_improvement(score, scores):
#                     scores = score
#                     candidate = {'type': 'lag', 'name': lg, 'model': model_test}
#                     improvement = True
#         if best_features["best_transforms"]:
#             for trans in best_features["best_transforms"]:
#                 trans_to_test = [x for x in best_features["best_transforms"] if x != trans]
#                 model_test = model.copy()
#                 model_test.lag_transform = trans_to_test
#                 model_test = model_update(model_test, df)
#                 score = validation(model_test, df)
#                 if is_elementwise_improvement(score, scores):
#                     scores = score
#                     candidate = {'type': 'transform', 'name': trans, 'model': model_test}
#                     improvement = True
#         if best_features["best_exogs"]:
#             for feat in best_features["best_exogs"]:
#                 # feat_to_test = [x for x in candidate_features if x != feat]
#                 df_test = df.drop(columns=feat)
#                 model_test = model.copy()
#                 model_test = model_update(model_test, df_test)
#                 score = validation(model_test, df_test)
#                 if is_elementwise_improvement(score, scores):
#                     scores = score
#                     candidate = {'type': 'exog', 'name': feat, 'model': model_test}
#                     improvement = True

#         # Update best features
#         if improvement and candidate['type']:
#             best_score = scores
#             if candidate['type'] == 'lag':
#                 best_features["best_lags"].remove(candidate['name'])
#             elif candidate['type'] == 'exog':
#                 best_features["best_exogs"].remove(candidate['name'])
#                 df = df.drop(columns=candidate['name'])
#             elif candidate['type'] == 'transform':
#                 best_features["best_transforms"].remove(candidate['name'])
#                 if not best_features["best_transforms"]:
#                     model.lag_transform = best_features["best_transforms"]
#                 else:
#                     model.lag_transform = None
#             model.A = candidate['model'].A
#             model.stds = candidate['model'].stds
#             model.LL = candidate['model'].LL
#             model.pi = candidate['model'].pi

#             if verbose:
#                 print(f"Removed {candidate['type']}: {candidate['name']} with score: {best_score} and loglik and BIC: {model.LL}, {model.BIC}")
#         else:
#             break  # No improvement

#         # Finalize model with best features
#         model_ = model.copy()
#         if lags_to_consider is not None and best_features["best_lags"]:
#             model_.lags = best_features["best_lags"]
#         if transformations is not None and best_features["best_transforms"]:
#             model_.lag_transform = best_features["best_transforms"]
#         model_ = model_update(model_, df)


#     if transformations is not None and best_features["best_transforms"]:
#         best_features["best_transforms"] = [trans.get_name() for trans in best_features["best_transforms"]]

#     return best_features, model_


# def hmm_mv_forward_feature_selection(df, target_col, n_folds = None, H = None, model = None, metrics = None,
#                                   lags_to_consider = None, candidate_features = None, transformations = None, 
#                                     step_size = None, starting_lags = None, starting_transforms = None,
#                                     validation_type = "cv", iterations = 10, verbose = False):
#     """
#     Performs forward lag selection for Vektor Autoregressive models and bidirectional ml models
#     Parameters:
#         df (pd.DataFrame): DataFrame containing the time series data.
#         target_col (str): The target column for accuracy evaluation.
#         n_folds (int): Number of folds for cross-validation.
#         H (int): Forecast horizon.
#         model: The forecasting model to be used.
#         metrics (list): List of metrics to evaluate the model.
#         lags_to_consider (dict): Dictionary of maximum lags for each variable.
#         candidate_features (list): List of candidate exogenous features.
#         transformations (list): List of transformations to consider.
#         step_size (int, optional): Step size for lag selection. Defaults to None.
#         starting_lags (dict, optional): Dictionary of starting lags for each variable. Defaults to None.
#         starting_transforms (dict, optional): Dictionary of starting transformations for each variable. Defaults to None.
#         verbose (bool, optional): Whether to print progress. Defaults to False.
#     Returns:
#         dict: Dictionary of best features for each variable.
#     """

#     # max_lag = sum(x for x in max_lags.values())
    
#     # lags = list(range(1, max_lags+1))
#     if lags_to_consider is None:
#         lags_to_consider = {}
#     if transformations is None:
#         transformations = {}

#     best_features = {"best_lags": {i: [] for i in lags_to_consider if lags_to_consider is not None}, "best_transforms": {i: [] for i in transformations if transformations is not None}, "best_exogs": []}
#     remaining_lags = {i:list(range(1, j+1)) for i, j in lags_to_consider.items()}
#     if starting_lags is not None:
#         for k, v in starting_lags.items():
#             all_lags = remaining_lags[k]
#             remaining_lags[k] = [x for x in all_lags if x not in v]
#             best_features["best_lags"][k].extend(v)

#     if lags_to_consider is not None:
#         model.lags = None # Start with no lags

#     # Keep original for feature add-back
#     df_orig = df.copy()

#     # Drop candidate features initially
#     if candidate_features:
#         df = df.drop(columns=candidate_features) # Drop candidate features to start with feature selection
#     if transformations is not None:
#         if starting_transforms is not None:
#             for k, v in starting_transforms.items():
#                 transformations[k] = [x for x in transformations if x not in v]
#                 best_features["best_transforms"][k].extend(v)
#             model.lag_transform = starting_transforms
#         else:
#             model.lag_transform = None # Start with no transformations


#     if validation_type == "cv":
#         if isinstance(metrics, list):
#             best_score = [float('inf')] * len(metrics)
#         else:
#             best_score = float('inf')
#     elif validation_type in ("BIC", "AIC"):
#         best_score = float('inf')
#     elif validation_type == "AIC_BIC":
#         best_score = [float('inf')] * 2
#     else:
#         raise ValueError("Invalid validation_type. Choose from 'cv', 'BIC', 'AIC', or 'AIC_BIC'.")

#     if isinstance(best_score, list):
#         def is_elementwise_improvement(score, best_s):
#             return all(s < b for s, b in zip(score, best_s))
#     else:
#         def is_elementwise_improvement(score, best_s):
#             return score < best_s

# # After each feature selection step iterate model to make sure parameters are updated like transition probabilities and stds
#     def model_update(model_test, df_test, iterations=iterations):
#         model_test.data_prep(df_test) # update data preparation because if new lags to be consistent with coefficients
#         model_test.compute_coeffs() # update model coefficients because of new lags
#         model_test.fit(df_test, n_iter=iterations) # update model parameters like transition probs and stds
#         return model_test
    

#     def validation(model_test, df_test):
#         if validation_type == "cv":
#             cv_result = hmm_mv_cross_validate(model = model_test, df=df_test, cv_split=n_folds, test_size=H,
#                                         metrics = metrics, step_size= step_size, n_iter=iterations)

#             if isinstance(metrics, list):
#                 score = cv_result[target_col].tolist()
#             else:
#                 score = cv_result[target_col].values[0]
#         elif validation_type == "BIC":
#             score = model_test.BIC
#         elif validation_type == "AIC":
#             score = model_test.AIC
#         elif validation_type == "AIC_BIC":
#             score = [model_test.AIC, model_test.BIC]
#         else:
#             raise ValueError("Invalid validation_type. Choose from 'cv', 'BIC', 'AIC', or 'AIC_BIC'.")

#         return score

    
#     # while max_lag>0:
#     while True:
#         improvement = False
#         candidate = {'target': None, 'type': None, 'name': None}
#         scores = best_score
#         if lags_to_consider is not None:
#             for k, lg in remaining_lags.items():
#                 for x in lg:
#                     model_test = model.copy()
#                     current_lag = {a:b for a, b in best_features['best_lags'].items()}
#                     current_lag[k] = best_features['best_lags'][k] + [x]
#                     current_lag[k].sort()
#                     model_test.lags = current_lag

#                     model_test = model_update(model_test, df)
#                     score = validation(model_test, df)

#                     if is_elementwise_improvement(score, scores):
#                         scores = score
#                         candidate = {'target': k, 'type': 'lag', 'name': x, 'model': model_test}
#                         improvement = True

#         # Test Exogenous Features
#         if candidate_features is not None:
#             for feat in candidate_features:
#                 df_test = df.copy()
#                 df_test[feat] = df_orig[feat]
#                 model_test = model.copy()
#                 model_test = model_update(model_test, df_test)
#                 score = validation(model_test, df_test)

#                 if is_elementwise_improvement(score, scores):
#                     scores = score
#                     candidate = {'target': None, 'type': 'exog', 'name': feat, 'model': model_test}
#                     improvement = True

#             # Test Transformations
#         if transformations is not None:
#             for k, trans in transformations.items():
#                 for t in trans:
#                     model_test = model.copy()
#                     lag_transform = (model_test.lag_transform[k] or []) + [t]
#                     model_test.lag_transform[k] = lag_transform
#                     model_test = model_update(model_test, df)
#                     score = validation(model_test, df)
#                     if is_elementwise_improvement(score, scores):
#                         scores = score
#                         candidate = {'target': k, 'type': 'transform', 'name': t, 'model': model_test}
#                         improvement = True

#         # Update best features
#         if improvement:
#             best_score = scores
#             if candidate['type'] == 'lag':
#                 best_features["best_lags"][candidate['target']].append(candidate['name']) # store lags by target
#                 remaining_lags[candidate['target']].remove(candidate['name'])
#             elif candidate['type'] == 'exog':
#                 best_features["best_exogs"].append(candidate['name'])
#                 candidate_features.remove(candidate['name'])
#                 df[candidate['name']] = df_orig[candidate['name']]
#             elif candidate['type'] == 'transform':
#                 best_features["best_transforms"][candidate['target']].append(candidate['name'])
#                 transformations[candidate['target']].remove(candidate['name'])
#                 if model.lag_transform is None:
#                     transform_dict = {candidate['target']: [candidate['name']]}
#                     model.lag_transform = transform_dict
#                 else:
#                     if candidate['target'] not in model.lag_transform:
#                         model.lag_transform[candidate['target']] = [candidate['name']]
#                     else:
#                         model.lag_transform[candidate['target']].append(candidate['name'])

#             model.A = candidate['model'].A
#             model.covs = candidate['model'].covs
#             model.LL = candidate['model'].LL
#             model.pi = candidate['model'].pi

#             if verbose:
#                 print(f"Added {candidate['type']}: {candidate['name']} with score: {best_score} and loglik and BIC: {model.LL}, {model.BIC}")
#         else:
#             break  # No improvement

#     # Finalize model with best features
#     model_ = model.copy()
#     if lags_to_consider is not None and best_features["best_lags"]:
#         model_.lags = best_features["best_lags"]
#     if transformations is not None and best_features["best_transforms"]:
#         model_.lag_transform = best_features["best_transforms"]

#     model_ = model_update(model_, df)


#     if transformations is not None:
#         for key, trans in best_features["best_transforms"].items():
#             if trans:  # only process non-empty lists
#                 best_features["best_transforms"][key] = [t.get_name() for t in trans]

#     return best_features, model_


# def hmm_mv_backward_feature_selection(df, target_col, n_folds = None, H = None, model = None, metrics = None,
#                                   lags_to_consider = None, candidate_features = None, transformations = None, 
#                                     step_size = None, validation_type = "cv", iterations = 10, 
#                                     verbose = False):
#     """
#     Performs backward lag selection for Regression models.
#     Parameters:
#         df (pd.DataFrame): DataFrame containing the time series data.
#         target_col (str): The target column for accuracy evaluation.
#         n_folds (int, optional): Number of cross-validation folds.
#         H (int, optional): Forecast horizon.
#         model: The forecasting model to be used.
#         metrics (list): List of metrics to evaluate the model.
#         step_size (int, optional): Step size for cross-validation. Defaults to None.
#         verbose (bool, optional): Whether to print progress. Defaults to False.
#     Returns:
#         dict: Dictionary of best features for each variable.

#     """

#     # remaining_lags = {i:list(range(1, j+1)) for i, j in lags_to_consider.items()}
#     # best_lags = {i:[] for i in max_lags}
#     best_features = {
#         "best_lags": {i: list(range(1, j+1)) for i, j in (lags_to_consider or {}).items()},
#         "best_exogs": candidate_features.copy() if candidate_features is not None else [],
#         "best_transforms": {i: j for i, j in (transformations or {}).items()}
# }
    
#     ## setting the full model
#     if lags_to_consider is not None:
#         model.lags = best_features["best_lags"]
#     if transformations is not None:
#         model.lag_transform = best_features["best_transforms"]
#     # exogenous variables should be in df before passing df

#     if validation_type == "cv":
#         if isinstance(metrics, list):
#             best_score = [float('inf')] * len(metrics)
#         else:
#             best_score = float('inf')
#     elif validation_type in ("BIC", "AIC"):
#         best_score = float('inf')
#     elif validation_type == "AIC_BIC":
#         best_score = [float('inf')] * 2
#     else:
#         raise ValueError("Invalid validation_type. Choose from 'cv', 'BIC', 'AIC', or 'AIC_BIC'.")

#     if isinstance(best_score, list):
#         def is_elementwise_improvement(score, best_s):
#             return all(s < b for s, b in zip(score, best_s))
#     else:
#         def is_elementwise_improvement(score, best_s):
#             return score < best_s

# # After each feature selection step iterate model to make sure parameters are updated like transition probabilities and stds
#     def model_update(model_test, df_test, iterations=iterations):
#         model_test.data_prep(df_test) # update data preparation because if new lags to be consistent with coefficients
#         model_test.compute_coeffs() # update model coefficients because of new lags
#         model_test.fit(df_test, n_iter=iterations) # update model parameters like transition probs and stds
#         return model_test
    

#     def validation(model_test, df_test):
#         if validation_type == "cv":
#             cv_result = hmm_mv_cross_validate(model = model_test, df=df_test, cv_split=n_folds, test_size=H,
#                                         metrics = metrics, step_size= step_size, n_iter=iterations)

#             if isinstance(metrics, list):
#                 score = cv_result[target_col].tolist()
#             else:
#                 score = cv_result[target_col].values[0]
#         elif validation_type == "BIC":
#             score = model_test.BIC
#         elif validation_type == "AIC":
#             score = model_test.AIC
#         elif validation_type == "AIC_BIC":
#             score = [model_test.AIC, model_test.BIC]
#         else:
#             raise ValueError("Invalid validation_type. Choose from 'cv', 'BIC', 'AIC', or 'AIC_BIC'.")

#         return score

    
#     while True:
#         improvement = False
#         candidate = {'target': None, 'type': None, 'name': None}
#         scores = best_score
#         if lags_to_consider is not None:
#             for targ_l, lags in best_features["best_lags"].items():
#                 for lg in lags:
#                     lags_to_test = {a:b for a, b in lags.items()}
#                     # Remove the current lag lg from current target
#                     lags_to_test[targ_l] = [x for x in lags if x != lg]
#                     lags_to_test[targ_l].sort()
#                     model_test = model.copy()
#                     model_test.lags = lags_to_test

#                     model_test = model_update(model_test, df)
#                     score = validation(model_test, df)
#                     if is_elementwise_improvement(score, scores):
#                         scores = score
#                         candidate = {'target': targ_l, 'type': 'lag', 'name': lg, 'model': model_test}
#                         improvement = True
#         if transformations is not None:
#             for targ_t, trans in best_features["best_transforms"].items():
#                 for tr in trans:
#                     trans_to_test = {a:b for a, b in best_features["best_transforms"].items()}
#                     trans_to_test[targ_t] = [x for x in trans if x != tr]
#                     model_test = model.copy()
#                     # model_test.lags = remaining_lags
#                     model_test.lag_transform = trans_to_test
#                     model_test = model_update(model_test, df)
#                     score = validation(model_test, df)
#                     if is_elementwise_improvement(score, scores):
#                         scores = score
#                         candidate = {'target': targ_t, 'type': 'transform', 'name': trans, 'model': model_test}
#                         improvement = True
#         if candidate_features is not None:
#             for feat in best_features["best_exogs"]:
#                 # feat_to_test = [x for x in candidate_features if x != feat]
#                 df_test = df.drop(columns=feat)
#                 model_test = model.copy()
#                 model_test = model_update(model_test, df_test)
#                 score = validation(model_test, df_test)
#                 if is_elementwise_improvement(score, scores):
#                     scores = score
#                     candidate = {'target': None, 'type': 'exog', 'name': feat, 'model': model_test}
#                     improvement = True

#         # Update best features
#         if improvement and candidate['type']:
#             best_score = scores
#             if candidate['type'] == 'lag':
#                 best_features["best_lags"][candidate['target']].remove(candidate['name'])
#             elif candidate['type'] == 'exog':
#                 best_features["best_exogs"].remove(candidate['name'])
#                 df = df.drop(columns=candidate['name'])
#             elif candidate['type'] == 'transform':
#                 best_features["best_transforms"][candidate['target']].remove(candidate['name'])
#                 if any(best_features["best_transforms"][key] for key in best_features["best_transforms"]):
#                     best_features["best_transforms"] = {k: v for k, v in best_features["best_transforms"].items() if not len(v) == 0}
#                     model.lag_transform = best_features["best_transforms"]
#                 else:
#                     model.lag_transform = None

#             model.A = candidate['model'].A
#             model.covs = candidate['model'].covs
#             model.LL = candidate['model'].LL
#             model.pi = candidate['model'].pi

#             if verbose:
#                 print(f"Removed {candidate['type']}: {candidate['name']} with score: {best_score} and loglik and BIC: {model.LL}, {model.BIC}")
#         else:
#             break  # No improvement

#     # Finalize model with best features
#     model_ = model.copy()
#     if lags_to_consider is not None and best_features["best_lags"]:
#         model_.lags = best_features["best_lags"]
#     if transformations is not None and best_features["best_transforms"]:
#         model_.lag_transform = best_features["best_transforms"]

#     model_ = model_update(model_, df)

#     # if transformations is not None and at least one key is not empty get their names
#     if transformations is not None:
#         for key, trans in best_features["best_transforms"].items():
#             if trans:  # only process non-empty lists
#                 best_features["best_transforms"][key] = [t.get_name() for t in trans]

#     return best_features, model_

# #------------------------------------------------------------------------------
# # Holt-Winters Exponential Smoothing Model Tuning
# #------------------------------------------------------------------------------

# def tune_ets(data, param_space, cv_splits, horizon, eval_metric, eval_num, step_size = None, verbose = False):
#     """
#     Tune ETS model hyperparameters using Hyperopt.

#     Args:
#         data (array-like): Time series data.
#         param_space (dict): Hyperparameter search space.
#         cv_splits (int): Number of cross-validation splits.
#         horizon (int): Forecast horizon.
#         step_size (int): Step size for time series cross-validation.
#         eval_metric (function): Evaluation metric function.
#         eval_num (int): Number of evaluations for hyperparameter tuning.
#         verbose (bool): Whether to print progress.
#     Returns:
#         tuple: Best model parameters and fit parameters.
#     """
#     from statsmodels.tsa.holtwinters import ExponentialSmoothing
#     from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
#     from hyperopt.pyll import scope
#     tscv = ParametricTimeSeriesSplit(n_splits=cv_splits, test_size=horizon, step_size=step_size)
#     # Define the objective function for hyperparameter tuning
#     def objective(params):
#         if (params.get("trend") != None) & (params.get("seasonal") != None): # Both trend and seasonal are specified
#             alpha = params.get('smoothing_level') # Smoothing level for the level component
#             beta = params.get('smoothing_trend') # Smoothing level for the trend component
#             gamma = params.get('smoothing_seasonal') # Smoothing level for the seasonal component
#             trend_type = params.get('trend') # Trend type
#             season_type = params.get('seasonal') # Seasonal type
#             S = params.get('seasonal_periods') # Seasonal periods
#             if params.get("damped_trend"): # Damped trend
#                 damped_bool = params.get("damped_trend")
#                 if damped_bool:
#                     damp_trend = params.get('damping_trend')
#                 else:
#                     damp_trend = None
#             else:
#                 damped_bool = params.get("damped_trend")
#                 damp_trend = None

#         elif (params.get("trend") != None) & (params.get("seasonal") == None): # Trend is specified, seasonal is not
#             alpha = params.get('smoothing_level')
#             beta = params.get('smoothing_trend')
#             gamma = None
#             trend_type = params.get('trend')
#             season_type = params.get('seasonal')
#             S=None
#             if params.get("damped_trend"):
#                 damped_bool = params.get("damped_trend")
#                 damp_trend = params.get('damping_trend')
#             else:
#                 damped_bool = params.get("damped_trend")
#                 damp_trend = None

#         elif (params.get("trend") == None) & (params.get("seasonal") != None): # Seasonal is specified, trend is not
#             alpha = params.get('smoothing_level')
#             beta = None
#             gamma = params.get('smoothing_seasonal')
#             trend_type = params.get('trend')
#             season_type = params.get('seasonal')
#             S = params.get('seasonal_periods')
#             if params.get("damped_trend"):
#                 damped_bool = False
#                 damp_trend = None
#             else:
#                 damped_bool = params.get("damped_trend")
#                 damp_trend = None
                
#         else: # Neither trend nor seasonal is specified
#             alpha = params.get('smoothing_level')
#             beta = None
#             gamma = None
#             trend_type = params.get('trend')
#             season_type = params.get('seasonal')
#             S = None
#             if params.get("damped_trend"):
#                 damped_bool = False
#                 damp_trend = None
#             else:
#                 damped_bool = params.get("damped_trend")
#                 damp_trend = None
            

#         metric = [] # List to store evaluation metrics
#         # Perform time series cross-validation
#         for train_index, test_index in tscv.split(data):
#             train, test = data[train_index], data[test_index]
#             # Fit the Holt-Winters model with the specified parameters
#             hw_fit = ExponentialSmoothing(train ,seasonal_periods=S , seasonal=season_type, trend=trend_type, damped_trend = damped_bool).fit(smoothing_level = alpha, 
#                                                                                                                       smoothing_trend = beta,
#                                                                                                                       smoothing_seasonal = gamma,
#                                                                                                 damping_trend=damp_trend)
            
#             hw_forecast = hw_fit.forecast(len(test))
#             forecast_filled = np.nan_to_num(hw_forecast, nan=0)

#             #Evaluate using the specified metric
#             if eval_metric.__name__ in ["MASE", "SMAE", "SRMSE", "RMSSE"]:
#                 accuracy = eval_metric(test,
#                                     forecast_filled,
#                                     train)
#             else:
#                 accuracy = eval_metric(
#                         test,
#                         forecast_filled,
#                     )

#             # accuracy = eval_metric(test, forecast_filled)
#             metric.append(accuracy)
            
#         score = np.mean(metric)
#         if verbose ==True:
#             print ("SCORE:", score)

#         return {'loss':score, 'status':STATUS_OK}
    
#     # Perform hyperparameter optimization using Hyperopt
#     trials = Trials()
    
#     best_hyperparams = fmin(fn = objective,
#                     space = param_space,
#                     algo = tpe.suggest,
#                     max_evals = eval_num,
#                     trials = trials)
#     best_params = space_eval(param_space, best_hyperparams)
#     model_params = {
#         "trend": best_params.get("trend"),
#         "seasonal_periods": best_params.get("seasonal_periods"),
#         "seasonal": best_params.get("seasonal"),
#         "damped_trend": best_params.get("damped_trend")
#     }
#     fit_params = {
#         "smoothing_level": best_params.get("smoothing_level"),
#         "smoothing_trend": best_params.get("smoothing_trend"),
#         "smoothing_seasonal": best_params.get("smoothing_seasonal"),
#         "damping_trend": best_params.get("damping_trend")
#     }

    
#     if set(model_params.keys()) == {"damped_trend"}: # if only damped_trend is in model_params, ensure it's False
#         model_params["damped_trend"] = False

#     # Remove all keys with value None in a single step
#     model_params = {k: v for k, v in model_params.items() if v is not None}
#     fit_params = {k: v for k, v in fit_params.items() if v is not None}
#     fit_params = {k: v for k, v in fit_params.items()
#                   if not (k == "damping_trend" and model_params.get("damped_trend") is False)}
#     return model_params, fit_params

# def cv_hmm_lag_tune(
#     model,
#     df,
#     cv_split,
#     test_size,
#     eval_metric,
#     lag_space=None,
#     step_size=None,
#     opt_horizon=None,
#     eval_num=100,
#     verbose=False,
# ):
#     """
#     Tune forecasting model hyperparameters using cross-validation and Bayesian optimization.

#     Parameters
#     ----------
#     model : object
#         Forecasting model object with .fit and .forecast methods and relevant attributes.
#     df : pd.DataFrame
#         Time series dataframe.
#     cv_split : int
#         Number of time series splits.
#     test_size : int
#         Size of test window for each split.
#     lag_space : dict
#         Hyperopt lag search space.
#     eval_metric : callable
#         Evaluation metric function.
#     step_size : int, optional
#         Step size for moving the window. Defaults to None (equal to test_size).
#     opt_horizon : int, optional
#         Evaluate only on last N points of each split. Defaults to None (all points).
#     eval_num : int, optional
#         Number of hyperopt evaluations. Defaults to 100.
#     verbose : bool, optional
#         Print progress. Defaults to False.

#     Returns
#     -------
#     dict
#         Best hyperparameter values found.
#     """
#     tscv = ParametricTimeSeriesSplit(n_splits=cv_split, test_size=test_size, step_size=step_size)
    
#     max_lag = lag_space 
#     space = {f"lag_{i}": hp.choice(f"lag_{i}", [0, 1]) for i in range(1, max_lag + 1)}
#     def objective(params):

#         selected_lags = [i for i in range(1, max_lag + 1) if params[f"lag_{i}"] == 1]
#         model_ = model.copy()
#         model_.lags = selected_lags

#                 # Optional: penalize too few lags
#         if len(selected_lags) < 1:
#             return {"loss": 1e6, "status": STATUS_OK}

#         metrics = []
#         for idx, (train_index, test_index) in enumerate(tscv.split(df)):
#             train, test = df.iloc[train_index], df.iloc[test_index]
#             x_test = test.drop(columns=[model.target_col])
#             y_test = np.array(test[model.target_col])
#             if idx == 0:
#                 model_.fit_em(train)
#             else:
#                 model_.fit(train)
            
#             y_pred = model_.forecast(len(y_test), x_test)

#             #Evaluate using the specified metric
#             if eval_metric.__name__ in ["MASE", "SMAE", "SRMSE", "RMSSE"]:
#                 score = eval_metric(y_test[-opt_horizon:] if opt_horizon else y_test,
#                                     y_pred[-opt_horizon:] if opt_horizon else y_pred,
#                                     train[model.target_col])
#             else:
#                 score = eval_metric(
#                         y_test[-opt_horizon:] if opt_horizon else y_test,
#                         y_pred[-opt_horizon:] if opt_horizon else y_pred,
#                     )
#             metrics.append(score)

#         mean_score = np.mean(metrics)
#         if verbose:
#             print("Score:", mean_score)
#         return {"loss": mean_score, "status": STATUS_OK}

#     trials = Trials()
#     best_hyperparams = fmin(
#         fn=objective,
#         space=space,
#         algo=tpe.suggest,
#         max_evals=eval_num,
#         trials=trials,
#     )

#     # Extract and sort lag values
#     best_lag_indexes = [value for key, value in sorted(((k, v) for k, v in best_hyperparams.items() if k.startswith("lag_")),
#                                                           key=lambda x: int(x[0].split("_")[1]))]
#     best_lag_values = [i for i in range(1, lag_space + 1) if best_lag_indexes[i-1]==1]
#     return best_lag_values

# #------------------------------------------------------------------------------
# # SARIMA Model Tuning
# #------------------------------------------------------------------------------

# def tune_sarima(y, d, D, season,p, q, P, Q, X=None):
#     """
#     Finds the best SARIMA parameters using AIC as the evaluation metric.
#     Args:
#         y (array-like): The time series data.
#         d (int): The non-seasonal differencing order.
#         d (int): The non-seasonal differencing order.
#         D (int): The seasonal differencing order.
#         season (int): The seasonal period.
#         p (int): Max range of non-seasonal AR orders to test.
#         q (int): Max range of non-seasonal MA orders to test.
#         P (int): Max range of seasonal AR orders to test.
#         Q (int): Max range of seasonal MA orders to test.
#         X (array-like, optional): Exogenous variables. Defaults to None.
#     Returns:
#         pd.DataFrame: A DataFrame containing the combinations of parameters and their corresponding AIC values.
#     """
#     if X is not None:
#         X = np.array(X, dtype = np.float64)
#     p_orders = range(0, p+1)
#     q_orders = range(0, q+1) # MA(q)
#     P_orders = range(0, P+1) # Seasonal autoregressive order.
#     Q_orders = range(0, Q+1) #Seasonal moving average order.
#     parameters = product(p_orders, q_orders, P_orders, Q_orders)
#     result = []
#     for param in tqdm_notebook(parameters):
#         try:
#             model = SARIMAX(endog=y, exog = X, order = (param[0], d, param[1]), seasonal_order= (param[2], D, param[3], season)).fit(disp = -1)
#         except:
#             continue
                            
#         aic = model.aic
#         result.append([param, aic])
#     result_df = pd.DataFrame(result)
#     result_df.columns = ["(p, q)x(P, Q)", "AIC"] 
#     result_df = result_df.sort_values("AIC", ascending = True) #Sort in ascending order, lower AIC is better
#     return result_df

# #------------------------------------------------------------------------------
# # ML tuning utility function
# #------------------------------------------------------------------------------


# def arima_cross_validate(model, df, target_col, cv_split, test_size, metrics,
#                          step_size=1, h_split_point=None):
#     """
#     Run cross-validation using time series splits.

#     Args:
#         model (class): ARIMA model instance. Supports ARIMA from statsforecast (Nixtla).
#         df (pd.DataFrame): Input data.
#         target_col (str): Name of the target column.
#         cv_split (int): Number of splits in TimeSeriesSplit.
#         test_size (int): Size of test window.
#         metrics (list): List of metric functions.
#         step_size (int): Step size for time series cross-validation.
#         h_split_point (int, optional): If provided, split the horizon into two parts for separate evaluation.
    
#     Returns:
#         pd.DataFrame: Performance metrics for CV.
#     """
#     tscv = ParametricTimeSeriesSplit(n_splits=cv_split, test_size=test_size, step_size=step_size)
#     metrics_dict = {m.__name__: [] for m in metrics}
#     if h_split_point is not None:
#         metrics_dict1 = {m.__name__: [] for m in metrics}
#         metrics_dict2 = {m.__name__: [] for m in metrics}
#     for train_index, test_index in tscv.split(df):
#         train, test = df.iloc[train_index], df.iloc[test_index]
#         x_test = test.drop(columns=[target_col]).to_numpy()
#         y_test = np.array(test[target_col])
#         y_train = train[target_col].to_numpy()
#         x_train = train.drop(columns=[target_col]).to_numpy()
#         forecasts = model.forecast(y=y_train, h=test_size, X=x_train, X_future=x_test)["mean"]
#         for m in metrics:
#             if m.__name__ in ["MASE", "SMAE", "SRMSE", "RMSSE"]:
#                 eval_val = m(y_test, forecasts, train[target_col])
#             else:
#                 eval_val = m(y_test, forecasts)
#             metrics_dict[m.__name__].append(eval_val)
#         if h_split_point is not None and isinstance(h_split_point, int):
#             y_test_1, y_test_2 = y_test[:h_split_point], y_test[h_split_point:]
#             bb_forecast_1, bb_forecast_2 = forecasts[:h_split_point], forecasts[h_split_point:]
#             for m in metrics:
#                 if m.__name__ in ['MASE', 'SMAE', 'SRMSE', 'RMSSE']:
#                     eval_val1 = m(y_test_1, bb_forecast_1, np.array(train[model.target_col]))
#                     eval_val2 = m(y_test_2, bb_forecast_2, np.array(train[model.target_col]))
#                 else:
#                     eval_val1 = m(y_test_1, bb_forecast_1)
#                     eval_val2 = m(y_test_2, bb_forecast_2)
#                 metrics_dict1[m.__name__].append(eval_val1)
#                 metrics_dict2[m.__name__].append(eval_val2)

#     overall_performance = [[m.__name__, np.mean(metrics_dict[m.__name__])] for m in metrics]
#     overall_performance = pd.DataFrame(overall_performance).rename(columns={0: "eval_metric", 1: "score"})
#     if h_split_point is not None and isinstance(h_split_point, int):
#         performance_1 = [[m.__name__, np.mean(metrics_dict1[m.__name__])] for m in metrics]
#         performance_2 = [[m.__name__, np.mean(metrics_dict2[m.__name__])] for m in metrics]
#         overall_performance = pd.DataFrame(overall_performance).rename(columns={0: "eval_metric", 1: "score"})
#         perf_1_df = pd.DataFrame(performance_1).rename(columns={0: "eval_metric", 1: f"score_before_{h_split_point}"})
#         perf_2_df = pd.DataFrame(performance_2).rename(columns={0: "eval_metric", 1: f"score_after_{h_split_point}"})
#         # merge all three dataframes
#         overall_performance = overall_performance.merge(perf_1_df, on="eval_metric").merge(perf_2_df, on="eval_metric")
#     return overall_performance

# def mv_cv_tune(
#     model,
#     df,
#     forecast_col,
#     cv_split,
#     test_size,
#     eval_metric,
#     param_space,
#     lag_space=None,
#     transform_space=None,
#     step_size=None,
#     opt_horizon=None,
#     eval_num=100,
#     verbose=False,
# ):
#     """
#     Tune forecasting model hyperparameters using cross-validation and Bayesian optimization.

#     Args:
#         model: Forecasting model object with .fit and .forecast methods and relevant attributes.
#         df (pd.DataFrame): Time series dataframe.
#         forecast_col (str): Name of the target variable to test.
#         cv_split (int): Number of time series splits.
#         test_size (int): Size of test window for each split.
#         step_size (int): Step size for moving the window. Defaults to None (equal to test_size).
#         param_space (dict): Hyperopt parameter search space. params for lags, differencing, etc. can be {'n_lag': (hp.choice('lag_y1', [1,2,3]), hp.choice('lag_y2', [1,2]))}
#         lag_space (dict, optional): Lag hyperparameter search space.
#         transform_space (dict, optional): Transformation hyperparameter search space.
#         eval_metric (callable): Evaluation metric function.
#         opt_horizon (int, optional): Evaluate only on last N points of each split. Defaults to None (all points).
#         eval_num (int, optional): Number of hyperopt evaluations. Defaults to 100.
#         verbose (bool, optional): Print progress. Defaults to False.

#     Returns:
#         dict: Best hyperparameter values found.
#     """

#     # target_cols = model.target_cols
#     tscv = ParametricTimeSeriesSplit(n_splits=cv_split, test_size=test_size, step_size=step_size)


#     def _set_model_params(params):
#         # Handle special model parameters that are not passed to model constructor
#         # and must be set directly on the forecasting model object
#         if "n_lag" in params:
#             if isinstance(params["n_lag"], dict):
#                 if model.n_lag is None:
#                     model.n_lag = {}
#                 # If n_lag is a dict, set it for each target column
#                 for target_col, lags in params["n_lag"].items():
#                     if isinstance(lags, int):
#                         model.n_lag[target_col] = list(range(1, lags + 1))
#                     elif isinstance(lags, list):
#                         model.n_lag[target_col] = lags

#         if "box_cox" in params:
#             if isinstance(params["box_cox"], dict):
#                 for target_col, box_cox in params["box_cox"].items():
#                     model.box_cox[target_col] = box_cox
#         if "box_cox_lmda" in params:
#             if isinstance(params["box_cox_lmda"], dict):
#                 for target_col, lamda in params["box_cox_lmda"].items():
#                     model.lamda[target_col] = lamda
#         if "box_cox_biasadj" in params:
#             if isinstance(params["box_cox_biasadj"], dict):
#                 for target_col, biasadj in params["box_cox_biasadj"].items():
#                     model.biasadj[target_col] = biasadj

#     def _get_model_params_for_fit(params):
#         # Exclude special parameters that should not be passed to the model constructor
#         skip_keys = {
#             "box_cox", "n_lag", "box_cox_lmda", "box_cox_biasadj"}
#         if lag_space is not None:
#             for var, max_lag in lag_space.items():
#                 skip_keys.update([f"{var}_lag_{i}" for i in range(1, max_lag + 1)])
#         if transform_space is not None:
#             for var, trans in transform_space.items():
#                 skip_keys.update([f"{var}_{t.get_name()}" for t in trans])

#         return {k: v for k, v in params.items() if k not in skip_keys}

#     if lag_space is not None:
#         lag_positions = {}
#         for var, max_lag in lag_space.items():
#             for i in range(1, max_lag + 1):
#                 lag_positions[f"{var}_lag_{i}"] = hp.choice(f"{var}_lag_{i}", [0, 1])
#         search_space = {**lag_positions, **param_space}
#     else:
#         search_space = {**param_space}

#     if transform_space is not None:
#         transform_positions = {}
#         for var, t in transform_space.items():
#             transform_positions[f"{var}_{t.get_name()}"] = hp.choice(t.get_name(), [0, 1])
#         search_space = {**transform_positions, **search_space}

#     def objective(params):
#         _set_model_params(params)
#         if isinstance(model.model, LinearRegression):
#             # For LinearRegression, we don't need to set model_params
#             model_params = None
#         else:
#             # For other models, get the parameters to set
#             model_params = _get_model_params_for_fit(params)

#         # Set lagged features
#         if lag_space is not None:
#             n_lag_dict = {}
#             for var, max_lag in lag_space.items():
#                 selected = [i for i in range(1, max_lag + 1) if params[f"{var}_lag_{i}"] == 1]
#                 if selected:   # keep only if some lags chosen
#                     n_lag_dict[var] = selected
#             model.n_lag = n_lag_dict
#             # penalize if no lag is selected for any target
#             if all(len(lags) == 0 for lags in n_lag_dict.values()):
#                 model.n_lag = None

#         # Set transformations
#         if transform_space is not None:
#             transform_dict = {}
#             for var, trans in transform_space.items():
#                 selected_trans = [t for t in trans if params[f"{var}_{t.get_name()}"] == 1]
#                 if selected_trans:  # keep only if some transformations chosen
#                     transform_dict[var] = selected_trans
#             model.transform = transform_dict
#             if all(len(t) == 0 for t in transform_dict.values()):
#                 model.transform = None

#         metrics = []
#         for train_index, test_index in tscv.split(df):
#             train, test = df.iloc[train_index], df.iloc[test_index]
#             x_test = test.drop(columns=model.target_cols)
#             y_test = np.array(test[forecast_col])

#             if model_params is not None:
#                 model.model.set_params(**model_params)
#             model.fit(train)

#             y_pred = model.forecast(len(y_test), x_test)[forecast_col]

#             #Evaluate using the specified metric
#             if eval_metric.__name__ in ["MASE", "SMAE", "SRMSE", "RMSSE"]:
#                 score = eval_metric(y_test[-opt_horizon:] if opt_horizon else y_test,
#                                     y_pred[-opt_horizon:] if opt_horizon else y_pred,
#                                     train[model.target_col])
#             else:
#                 score = eval_metric(
#                         y_test[-opt_horizon:] if opt_horizon else y_test,
#                         y_pred[-opt_horizon:] if opt_horizon else y_pred,
#                     )
#             metrics.append(score)

#         mean_score = np.mean(metrics)
#         if verbose:
#             print("Score:", mean_score)
#         return {"loss": mean_score, "status": STATUS_OK}

#     trials = Trials()
#     best_hyperparams = fmin(
#         fn=objective,
#         space=search_space,
#         algo=tpe.suggest,
#         max_evals=eval_num,
#         trials=trials,
#     )

#     model.tuned_params = [
#         space_eval(param_space, {k: v[0] for k, v in t["misc"]["vals"].items()})
#         for t in trials.trials
#     ]

#     best_lags = {}
#     if lag_space is not None:
#         for var, max_lag in lag_space.items():
#             chosen = [i for i in range(1, max_lag + 1) if best_hyperparams[f"{var}_lag_{i}"] == 1]
#             best_lags[var] = chosen
#     best_transforms = {}
#     if transform_space is not None:
#         for var, trans in transform_space.items():
#             chosen = [t.get_name() for t in trans if best_hyperparams[f"{var}_{t.get_name()}"] == 1]
#             best_transforms[var] = chosen

#     return space_eval(param_space, best_hyperparams), best_lags, best_transforms

# def cv_lag_tune(
#     model,
#     df,
#     cv_split,
#     test_size,
#     eval_metric,
#     lag_space=None,
#     step_size=None,
#     opt_horizon=None,
#     eval_num=100,
#     verbose=False,
# ):
#     """
#     Tune forecasting model hyperparameters using cross-validation and Bayesian optimization.

#     Parameters
#     ----------
#     model : object
#         Forecasting model object with .fit and .forecast methods and relevant attributes.
#     df : pd.DataFrame
#         Time series dataframe.
#     cv_split : int
#         Number of time series splits.
#     test_size : int
#         Size of test window for each split.
#     lag_space : dict
#         Hyperopt lag search space.
#     eval_metric : callable
#         Evaluation metric function.
#     step_size : int, optional
#         Step size for moving the window. Defaults to None (equal to test_size).
#     opt_horizon : int, optional
#         Evaluate only on last N points of each split. Defaults to None (all points).
#     eval_num : int, optional
#         Number of hyperopt evaluations. Defaults to 100.
#     verbose : bool, optional
#         Print progress. Defaults to False.

#     Returns
#     -------
#     dict
#         Best hyperparameter values found.
#     """
#     tscv = ParametricTimeSeriesSplit(n_splits=cv_split, test_size=test_size, step_size=step_size)
#     max_lag = lag_space
#     space = {f"lag_{i}": hp.choice(f"lag_{i}", [0, 1]) for i in range(1, max_lag + 1)}
#     def objective(params):

#         selected_lags = [i for i in range(1, max_lag + 1) if params[f"lag_{i}"] == 1]
#         model_ = model.copy()
#         model_.n_lag = selected_lags

#                 # Optional: penalize too few lags
#         if len(selected_lags) < 1:
#             return {"loss": 1e6, "status": STATUS_OK}

#         metrics = []
#         for train_index, test_index in tscv.split(df):
#             train, test = df.iloc[train_index], df.iloc[test_index]
#             x_test = test.drop(columns=[model.target_col])
#             y_test = np.array(test[model.target_col])
#             model_.fit(train)

#             y_pred = model_.forecast(len(y_test), x_test)

#             #Evaluate using the specified metric
#             if eval_metric.__name__ in ["MASE", "SMAE", "SRMSE", "RMSSE"]:
#                 score = eval_metric(y_test[-opt_horizon:] if opt_horizon else y_test,
#                                     y_pred[-opt_horizon:] if opt_horizon else y_pred,
#                                     train[model.target_col])
#             else:
#                 score = eval_metric(
#                         y_test[-opt_horizon:] if opt_horizon else y_test,
#                         y_pred[-opt_horizon:] if opt_horizon else y_pred,
#                     )
#             metrics.append(score)

#         mean_score = np.mean(metrics)
#         if verbose:
#             print("Score:", mean_score)
#         return {"loss": mean_score, "status": STATUS_OK}

#     trials = Trials()
#     best_hyperparams = fmin(
#         fn=objective,
#         space=space,
#         algo=tpe.suggest,
#         max_evals=eval_num,
#         trials=trials,
#     )

#     # Extract and sort lag values
#     best_lag_indexes = [value for key, value in sorted(((k, v) for k, v in best_hyperparams.items() if k.startswith("lag_")),
#                                                           key=lambda x: int(x[0].split("_")[1]))]
#     best_lag_values = [i for i in range(1, lag_space + 1) if best_lag_indexes[i-1]==1]
#     return best_lag_values

# def prob_param_forecasts(model, H, train_df, test_df=None):
#     prob_forecasts = []
#     for params in model.tuned_params:
#         if ('n_lag' in params) |('differencing_number' in params)|('box_cox' in params)|('box_cox_lmda' in params)|('box_cox_biasadj' in params):
#             if ('n_lag' in params):
#                 if type(params["n_lag"]) is tuple:
#                     model.n_lag = list(params["n_lag"])
#                 else:
#                     model.n_lag = range(1, params["n_lag"]+1)

#             if ('differencing_number' in params):
#                 model.difference = params["differencing_number"]
#             if ('box_cox' in params):
#                 model.box_cox = params["box_cox"]
#             if ('box_cox_lmda' in params):
#                 model.lmda = params["box_cox_lmda"]

#             if ('box_cox_biasadj' in params):
#                 model.biasadj = params["box_cox_biasadj"]

        
#         if model.model.__name__ != 'LinearRegression':
#             model_params = {k: v for k, v in params.items() if (k not in ["box_cox", "n_lag", "box_cox_lmda", "box_cox_biasadj"])}
#             model.fit(train_df, model_params)
#         else:
#             model.fit(train_df)
#         if test_df is not None:
#             forecasts = model.forecast(H, test_df)
#         else:
#             forecasts = model.forecast(H)

#         prob_forecasts.append(forecasts)
#     prob_forecasts = np.row_stack(prob_forecasts)
#     prob_forecasts=pd.DataFrame(prob_forecasts)
#     prob_forecasts.columns = ["horizon_"+str(i+1) for i in prob_forecasts.columns]
#     return prob_forecasts